In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Create the revised Figure 5 WSCI graphics.

Outputs
-------
1. Main Figure 5:
   panel a = global WSCI map
   panel b = nine local zoom-in maps arranged as a 3 x 3 block

2. Standalone supplementary figure:
   river sediment flux versus WSCI (the former panel c), with no panel letter

3. Arctic-exclusion sensitivity figure:
   compares the complete valid WSCI domain with the primary analysis restricted
   to latitudes <= 66 degrees N. The comparison uses pixel-area weighting for
   statements about the fraction of coastal area that is wave-sensitive.

The plotting logic, raster handling, land masks, scale bars, global
histogram/CDF inset, and local-panel styling are retained from the original
notebook wherever possible. Panel-a rivers remain Natural Earth for efficient
global rendering. Local zoom rivers use the high-resolution GRIT v1.0
simplified river network, with the existing Louisiana custom river shapefile
retained for panel 1 and Natural Earth used only as a per-panel fallback.

The default WSCI palette is a custom light-to-dark teal/indigo sequential
colormap. Low WSCI values remain visually quiet, whereas high WSCI values
become progressively darker and more saturated. This avoids imposing a
red/blue "hot-cold" interpretation on WSCI while retaining strong visual
hierarchy on the global and local maps. Several alternative palettes remain
selectable from the quick-control block near the top of this file.
"""

import os
import gc
import csv
import warnings
from typing import Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from rasterio.windows import from_bounds as window_from_bounds

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle
from matplotlib.collections import LineCollection
import matplotlib.patheffects as pe

warnings.filterwarnings("ignore", category=UserWarning)

# =============================================================================
# User-adjustable settings
# =============================================================================

# -----------------------------------------------------------------------------
# QUICK CONTROLS: panel labels and WSCI colormap
# -----------------------------------------------------------------------------
# Main Figure 5 contains only:
#   panel a = global WSCI map
#   panel b = 3 x 3 local zoom panels
# The former river-flux panel is saved as a separate supplementary figure
# and therefore has no panel letter.

PANEL_A_LABEL = "a"
PANEL_B_LABEL = "b"

# WSCI colormap choices.
#
# Change only COLORMAP_CHOICE to switch the palette throughout the entire figure
# (global map, local zoom panels, colorbar, and WSCI histogram inset).
#
# Recommended default:
#   "nature_teal_indigo"
#       Warm ivory -> muted mint -> teal -> blue -> deep indigo.
#       This is intentionally NOT a red/blue diverging palette. WSCI is treated
#       as an ordered magnitude: lower values are pale and quiet, while the
#       strongest coupling approaches a dark indigo. This generally works well
#       against the light-gray land background and avoids a "temperature map"
#       interpretation.
#
# Other custom publication-oriented choices:
#   "nature_emerald_navy"
#       Pale sage -> green/teal -> navy. Slightly cooler and cleaner.
#   "nature_lavender"
#       Pale gray-lilac -> muted blue-violet -> deep plum. More distinctive from
#       the light-blue river network and useful when teal feels too close to rivers.
#
# Standard Matplotlib alternatives are retained for quick comparison.
# "previous" reproduces the former RdYlBu_r palette exactly.
COLORMAP_CHOICE = "previous"

COLORMAP_OPTIONS = {
    # -------------------------------------------------------------------------
    # Custom publication-oriented palettes
    # -------------------------------------------------------------------------
    # Positions are normalized from 0 (DATA_VMIN) to 1 (DATA_VMAX).
    # The scientific normalization remains linear; only the perceptual color
    # progression is customized.
    "nature_teal_indigo": {
        "type": "custom",
        "name": "wsci_nature_teal_indigo",
        "colors": [
            (0.00, "#F2EBD5"),  # warm ivory / very low WSCI
            (0.18, "#D7E4D6"),  # pale sage
            (0.38, "#9BC9B8"),  # muted mint
            (0.58, "#58A3A0"),  # teal
            (0.76, "#3D788F"),  # blue-teal
            (0.90, "#45558A"),  # indigo-blue
            (1.00, "#352D67"),  # deep indigo / highest WSCI
        ],
    },

    "nature_emerald_navy": {
        "type": "custom",
        "name": "wsci_nature_emerald_navy",
        "colors": [
            (0.00, "#F1F0D8"),  # pale neutral
            (0.20, "#C8DDB8"),  # sage
            (0.40, "#85BD9A"),  # green
            (0.60, "#4C9A8C"),  # emerald-teal
            (0.78, "#377686"),  # blue-teal
            (0.90, "#31556F"),  # slate blue
            (1.00, "#25354F"),  # deep navy
        ],
    },

    "nature_lavender": {
        "type": "custom",
        "name": "wsci_nature_lavender",
        "colors": [
            (0.00, "#F1EFE7"),  # warm off-white
            (0.20, "#DDDCE7"),  # pale lavender gray
            (0.40, "#B9B8D6"),  # muted lavender
            (0.60, "#8E91BE"),  # blue-violet
            (0.78, "#686FA5"),  # muted indigo
            (0.90, "#514E86"),  # deep violet
            (1.00, "#392D63"),  # dark plum-indigo
        ],
    },

    # -------------------------------------------------------------------------
    # Standard Matplotlib palettes
    # -------------------------------------------------------------------------
    "viridis": {"type": "matplotlib", "name": "viridis"},
    "cividis": {"type": "matplotlib", "name": "cividis"},
    "magma": {"type": "matplotlib", "name": "magma"},
    "inferno": {"type": "matplotlib", "name": "inferno"},
    "plasma": {"type": "matplotlib", "name": "plasma"},
    "YlGnBu": {"type": "matplotlib", "name": "YlGnBu"},
    "GnBu": {"type": "matplotlib", "name": "GnBu"},
    "PuBuGn": {"type": "matplotlib", "name": "PuBuGn"},
    "BuPu": {"type": "matplotlib", "name": "BuPu"},
    "cubehelix": {"type": "matplotlib", "name": "cubehelix"},

    # Exact palette used in the earlier version.
    "previous": {"type": "matplotlib", "name": "RdYlBu_r"},
}

# Used only when LABEL_LOCAL_BLOCK_ONLY = False.
# Normally keep LABEL_LOCAL_BLOCK_ONLY = True so the nine local maps are one panel b.
LOCAL_PANEL_LABELS_WHEN_SEPARATE = list("bcdefghij")

# -----------------------------------------------------------------------------
# Thread / environment settings
# -----------------------------------------------------------------------------
NUM_THREADS = 6
os.environ["GDAL_NUM_THREADS"] = str(NUM_THREADS)
os.environ["OMP_NUM_THREADS"] = str(NUM_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(NUM_THREADS)
os.environ["MKL_NUM_THREADS"] = str(NUM_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(NUM_THREADS)

# -----------------------------------------------------------------------------
# Input and output paths
# -----------------------------------------------------------------------------
INPUT_TIF = "/projectnb/nasa-marsh/Hangjie/cloud/saltmarsh_lossgain/processed/correlation_matrix_globalcolor_mask.tif"
CSV_FILE_PATH = "/projectnb/nasa-marsh/Hangjie/cloud/sedimentflux_dethier/output_matrix.csv"
LOUISIANA_RIVER_SHP_PATH = "/projectnb/nasa-marsh/Hangjie/SWOT/shp/NGM_Rivers/America_River.shp"
GRIT_RIVER_GPKG_PATH = "/projectnb/nasa-marsh/Hangjie/Louisiana/cedric/river_shp/GRITv1.0_segments_simple_GLOBAL_EPSG4326.gpkg"

OUTPUT_DIR = "/projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/"
DATA_DIR = os.path.join(OUTPUT_DIR, "data")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# The primary manuscript analysis excludes only points north of 66 degrees N.
# This follows the requested data-support criterion and leaves the Southern
# Hemisphere unchanged.
MAX_ANALYSIS_LAT = 66.0

# Main Figure 5 = panel a + panel b only.
MAIN_OUTPUT_PNG = os.path.join(OUTPUT_DIR, "Figure5_WSCI_global_zoom.png")

# Former panel c, now a standalone supplementary figure with no panel letter.
SUPPLEMENT_OUTPUT_PNG = os.path.join(OUTPUT_DIR, "FigureS_WSCI_river_sediment_flux.png")

# New sensitivity figure comparing the full domain with the <=66N analysis.
ARCTIC_SENSITIVITY_OUTPUT = os.path.join(
    OUTPUT_DIR, "FigureS_Arctic_exclusion_WSCI_sensitivity.jpg"
)

OUTPUT_STATS_CSV = os.path.join(
    DATA_DIR, "Figure5_global_WSCI_0p5deg_statistics_latLE66.csv"
)
OUTPUT_FLUX_STATS_CSV = os.path.join(DATA_DIR, "FigureS_river_sediment_flux_statistics.csv")
ARCTIC_WSCI_SUMMARY_CSV = os.path.join(DATA_DIR, "Arctic_exclusion_WSCI_summary.csv")
MANUSCRIPT_SUMMARY_CSV = os.path.join(DATA_DIR, "Arctic_exclusion_summary_for_manuscript.csv")
OUTPUT_NOTES = os.path.join(DATA_DIR, "Figure5_runtime_notes.txt")

SAVE_PNG = True
FIG_DPI = 400

# Arctic-exclusion sensitivity figure controls.
ARCTIC_SENSITIVITY_FIGSIZE = (10.2, 4.0)

# Dedicated font sizes for the Arctic-exclusion supplementary figure.
# These are tuned separately from the main figure so that the text appears
# visually consistent with the manuscript figures despite the smaller canvas.
ARCTIC_AXIS_LABEL_FONT_SIZE = 8.6 * FONT_SCALE
ARCTIC_TICK_FONT_SIZE = 7.2 * FONT_SCALE
ARCTIC_LEGEND_FONT_SIZE = 7.3 * FONT_SCALE
ARCTIC_VALUE_FONT_SIZE = 7.0 * FONT_SCALE
ARCTIC_PANEL_LABEL_FONT_SIZE = 9.4 * FONT_SCALE
ARCTIC_TITLE_FONT_SIZE = 7.4 * FONT_SCALE

ARCTIC_DIAGNOSTIC_N_BINS = 400
ARCTIC_DOMAIN_LABEL_FULL = "Full valid domain"
ARCTIC_DOMAIN_LABEL_CUTOFF = r"$\leq 66^\circ$N"
ARCTIC_LINEWIDTH = 1.8
ARCTIC_MARKER_SIZE = 6.0

# -----------------------------------------------------------------------------
# Main Figure 5 layout
# -----------------------------------------------------------------------------
MAIN_FIGURE_WIDTH_INCH = 16.0
MAIN_FIGURE_SIDE_MARGIN_FRAC = 0.018
MAIN_FIGURE_TOP_MARGIN_FRAC = 0.985
MAIN_FIGURE_BOTTOM_MARGIN_FRAC = 0.030

# Approximate Robinson global-panel width / height ratio.
ROBINSON_PANEL_WIDTH_TO_HEIGHT = 1.972

# The nine local maps are arranged 3 x 3.
LOCAL_GRID_NROWS = 3
LOCAL_GRID_NCOLS = 3
LOCAL_GRID_HSPACE = 0.040
LOCAL_GRID_WSPACE = 0.045

# Physical white-space gap between global panel a and local panel b.
PANEL_A_TO_B_GAP_INCH = 0.30

# Standalone supplementary river-flux figure size.
SUPPLEMENT_FIGURE_WIDTH_INCH = 9.0
SUPPLEMENT_FIGURE_HEIGHT_INCH = 6.0
SUPPLEMENT_LEFT_MARGIN = 0.16
SUPPLEMENT_RIGHT_MARGIN = 0.97
SUPPLEMENT_BOTTOM_MARGIN = 0.16
SUPPLEMENT_TOP_MARGIN = 0.97

# -----------------------------------------------------------------------------
# Font and text controls
# Change FONT_SCALE once to enlarge or shrink all font sizes together.
# -----------------------------------------------------------------------------
FONT_FAMILY = "serif"
FONT_SCALE = 2.00

BASE_FONT_SIZE = 11 * FONT_SCALE
AXIS_LABEL_FONT_SIZE = 11 * FONT_SCALE
TICK_LABEL_FONT_SIZE = 8 * FONT_SCALE
PANEL_LABEL_FONT_SIZE = 16 * FONT_SCALE
GLOBAL_MARKER_LABEL_FONT_SIZE = 11 * FONT_SCALE
LOCAL_TITLE_FONT_SIZE = 10 * FONT_SCALE
SCALEBAR_FONT_SIZE = 9 * FONT_SCALE
COLORBAR_LABEL_FONT_SIZE = 10 * FONT_SCALE
COLORBAR_TICK_FONT_SIZE = 9 * FONT_SCALE
INSET_AXIS_LABEL_FONT_SIZE = 7.5 * FONT_SCALE
INSET_TICK_FONT_SIZE = 7 * FONT_SCALE
PANEL_C_AXIS_LABEL_FONT_SIZE = 12 * FONT_SCALE
PANEL_C_TICK_FONT_SIZE = 10 * FONT_SCALE
PANEL_C_LEGEND_FONT_SIZE = 10 * FONT_SCALE

plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": AXIS_LABEL_FONT_SIZE,
    "axes.labelsize": AXIS_LABEL_FONT_SIZE,
    "xtick.labelsize": TICK_LABEL_FONT_SIZE,
    "ytick.labelsize": TICK_LABEL_FONT_SIZE,
    "figure.titlesize": 16 * FONT_SCALE,
})

# Panel-label positions in axes coordinates.
PANEL_A_LABEL_POS = (-0.04, 0.985)
PANEL_B_LABEL_POS = (-0.13, 1.155)

# If True, label only the entire 3 x 3 local block using PANEL_B_LABEL.
# If False, each local zoom panel receives an individual lower-case letter.
LABEL_LOCAL_BLOCK_ONLY = True

# -----------------------------------------------------------------------------
# WSCI color settings
# -----------------------------------------------------------------------------
DATA_LABEL = "WSCI"
DATA_VMIN = 0.6
DATA_VMAX = 1.0
DATA_NORM = Normalize(vmin=DATA_VMIN, vmax=DATA_VMAX, clip=False)

if COLORMAP_CHOICE not in COLORMAP_OPTIONS:
    raise ValueError(
        f"Unknown COLORMAP_CHOICE={COLORMAP_CHOICE!r}. "
        f"Choose from {list(COLORMAP_OPTIONS)}"
    )


def build_wsci_colormap(choice: str):
    """
    Build the selected WSCI colormap.

    Custom palettes use explicit normalized color positions. Scientific
    normalization remains linear (DATA_VMIN-DATA_VMAX); only the visual
    progression changes. The default palette uses decreasing lightness and
    increasing saturation so that high WSCI is emphasized without implying
    a red/blue or hot/cold physical meaning.
    """
    spec = COLORMAP_OPTIONS[choice]

    if spec["type"] == "custom":
        cmap = LinearSegmentedColormap.from_list(
            spec["name"],
            spec["colors"],
            N=256,
        )
    elif spec["type"] == "matplotlib":
        cmap = cm.get_cmap(spec["name"])
        try:
            cmap = cmap.copy()
        except AttributeError:
            pass
    else:
        raise ValueError(f"Unsupported colormap specification: {spec}")

    # Transparent masked pixels preserve the white ocean/no-data background.
    cmap.set_bad((1.0, 1.0, 1.0, 0.0))

    # Values outside the displayed range keep the endpoint colors unless they
    # are explicitly masked with MASK_VALUES_OUTSIDE_COLOR_RANGE = True.
    cmap.set_under(cmap(0.0))
    cmap.set_over(cmap(1.0))
    return cmap


DATA_CMAP = build_wsci_colormap(COLORMAP_CHOICE)
DATA_CMAP_NAME = DATA_CMAP.name

# False preserves the original raster values below 0.6 and maps them to the
# low end of the selected colormap, matching the previous plotting logic.
MASK_VALUES_OUTSIDE_COLOR_RANGE = False

# -----------------------------------------------------------------------------
# Panel-a WSCI histogram + CDF inset controls
# Position is relative to panel-a axes: [left, bottom, width, height].
# -----------------------------------------------------------------------------
SHOW_WSCI_INSET = True
WSCI_INSET_AXES = [0.058, 0.100, 0.175, 0.235]
WSCI_HIST_RANGE = (-1.0, 1.0)
WSCI_HIST_N_BINS = 80
WSCI_HIST_THRESHOLD = 0.6
WSCI_HIST_BELOW_THRESHOLD_COLOR = None
WSCI_HIST_EDGE_COLOR = "none"
WSCI_HIST_ALPHA = 0.95
WSCI_HIST_USE_PERCENT = True
WSCI_CDF_LINE_COLOR = "black"
WSCI_CDF_LINEWIDTH = 1.35
WSCI_THRESHOLD_LINE_COLOR = "black"
WSCI_THRESHOLD_LINESTYLE = "--"
WSCI_THRESHOLD_LINEWIDTH = 1.0
WSCI_INSET_FACE_COLOR = "white"
WSCI_INSET_ALPHA = 0.92
WSCI_INSET_XTICKS = [-1.0, 0.0, 0.6, 1.0]
WSCI_INSET_YLABEL = "Frequency (%)"
WSCI_INSET_CDF_YLABEL = "Cumulative (%)"
WSCI_INSET_XLABEL = "WSCI"
WSCI_INSET_SPINE_LINEWIDTH = 0.6
WSCI_INSET_GRID = False

# -----------------------------------------------------------------------------
# Global panel scatter settings
# -----------------------------------------------------------------------------
GLOBAL_SCATTER_PIXEL_STEP = 1
GLOBAL_SCATTER_MARKER = "s"
GLOBAL_SCATTER_SIZE = 2.8
GLOBAL_SCATTER_ALPHA = 1.0
GLOBAL_SCATTER_ZORDER = 2

# -----------------------------------------------------------------------------
# Global rivers and land settings
# -----------------------------------------------------------------------------
GLOBAL_NE_SCALE = "50m"
LOCAL_NE_SCALE = "10m"
GLOBAL_RIVER_LINEWIDTH = 0.42
GLOBAL_RIVER_ALPHA = 0.90
LOCAL_RIVER_LINEWIDTH = 0.80
LOCAL_RIVER_ALPHA = 0.90
RIVER_COLOR = "#86c5e8"

# Local-river source strategy:
#   * panel a: Natural Earth, because it is a global overview
#   * panel b, site 1 Louisiana: existing custom Louisiana river shapefile
#   * panel b, sites 2-9: GRIT v1.0 simplified global river network
#   * if a GRIT bbox query unexpectedly returns no lines: Natural Earth fallback
#
# GRIT is read with a spatial bbox filter separately for each local panel.
# This avoids loading the entire global GeoPackage into memory.
REQUIRE_GRIT_LOCAL_RIVERS = True
GRIT_GPKG_LAYER = None  # None = auto-detect the first/most plausible layer
GRIT_BBOX_PAD_FRACTION = 0.015
SHOW_LOCAL_RIVER_DIAGNOSTICS = True
USE_LINECOLLECTION_FOR_LOCAL_RIVERS = True

BACKGROUND_COLOR = "white"
LAND_COLOR = "#d9d9d9"
LAND_EDGE_COLOR = "#6a6a6a"
LAND_EDGE_LINEWIDTH_GLOBAL = 0.40
LAND_EDGE_LINEWIDTH_LOCAL = 0.70

GLOBAL_LAND_BACKGROUND_RES_DEG = 0.25
GLOBAL_LAND_MASK_ALL_TOUCHED = False
LOCAL_LAND_MASK_ALL_TOUCHED = True

# -----------------------------------------------------------------------------
# Global boxes and markers
# -----------------------------------------------------------------------------
SHOW_GLOBAL_MARKERS = True
SHOW_GLOBAL_ZOOM_BOXES = True
GLOBAL_MARKER_SIZE = 2
GLOBAL_MARKER_FACE = "white"
GLOBAL_MARKER_EDGE = "black"
GLOBAL_MARKER_LINEWIDTH = 0
GLOBAL_BOX_COLOR = "black"
GLOBAL_BOX_LINEWIDTH = 1.25
GLOBAL_BOX_LINESTYLE = "--"

# Default projected offset used only when a site id is not listed below.
GLOBAL_MARKER_LABEL_DX = 0.02
GLOBAL_MARKER_LABEL_DY = 0.02

# Per-site manual label offsets for panel a.
# Site order:
# 1 Louisiana
# 2 Yukon River Delta
# 3 Rio de la Plata
# 4 Severn
# 5 Nile
# 6 Rufiji
# 7 Ganges-Brahmaputra-Meghna
# 8 Yellow River
# 9 Gulf of Carpentaria
GLOBAL_SITE_LABEL_LONLAT_OFFSETS_DEG = {
    1: (0.55, 0.42),
    2: (0.70, 0.40),
    3: (0.65, 0.42),
    4: (0.34, 0.25),
    5: (0.48, 0.34),
    6: (0.52, 0.34),
    7: (0.58, 0.38),
    8: (0.42, 0.30),
    9: (0.52, 0.34),
}
GLOBAL_SITE_LABEL_PROJECTED_OFFSETS = {
    1: (0.000, 0.000),
    2: (0.000, 0.000),
    3: (0.000, 0.000),
    4: (0.000, 0.000),
    5: (0.000, 0.000),
    6: (0.000, 0.000),
    7: (0.000, 0.000),
    8: (0.000, 0.000),
    9: (0.000, 0.000),
}

GLOBAL_SITE_LABEL_BOX_FACE = "white"
GLOBAL_SITE_LABEL_BOX_EDGE = "black"
GLOBAL_SITE_LABEL_BOX_ALPHA = 0.90
GLOBAL_SITE_LABEL_BOX_LINEWIDTH = 0.05
GLOBAL_SITE_LABEL_BOX_PAD = 0.28

# Panel-a zoom-box controls.
GLOBAL_BOX_EXPAND_FRACTION = 0.18
GLOBAL_BOX_EXTENT_OVERRIDES = {i: None for i in range(1, 10)}

# -----------------------------------------------------------------------------
# Local title settings
# -----------------------------------------------------------------------------
SHOW_LOCAL_TITLES = True
LOCAL_TITLE_AXES_X = 0.965
LOCAL_TITLE_AXES_Y = 0.955
LOCAL_TITLE_HA = "right"
LOCAL_TITLE_VA = "top"
LOCAL_TITLE_BOX_ALPHA = 0.72
LOCAL_TITLE_BOX_PAD = 1.5

LOCAL_TITLE_AXES_POS_OVERRIDES = {
    i: (LOCAL_TITLE_AXES_X, LOCAL_TITLE_AXES_Y) for i in range(1, 10)
}

# -----------------------------------------------------------------------------
# Local panel site-number settings
# -----------------------------------------------------------------------------
SHOW_LOCAL_SITE_SYMBOL = False
SHOW_LOCAL_SITE_NUMBER = True

LOCAL_SITE_NUMBER_AXES_X = 0.055
LOCAL_SITE_NUMBER_AXES_Y = 0.905
LOCAL_SITE_NUMBER_AXES_POS_OVERRIDES = {
    i: (LOCAL_SITE_NUMBER_AXES_X, LOCAL_SITE_NUMBER_AXES_Y) for i in range(1, 10)
}

LOCAL_SITE_NUMBER_FONT_SIZE = 11 * FONT_SCALE
LOCAL_SITE_NUMBER_BOX_FACE = "white"
LOCAL_SITE_NUMBER_BOX_EDGE = "black"
LOCAL_SITE_NUMBER_BOX_ALPHA = 0.88
LOCAL_SITE_NUMBER_BOX_LINEWIDTH = 0.55
LOCAL_SITE_NUMBER_BOX_PAD = 0.24

# Kept for backward compatibility. Not used when SHOW_LOCAL_SITE_SYMBOL = False.
LOCAL_SITE_SYMBOL_AXES_X = 0.055
LOCAL_SITE_SYMBOL_AXES_Y = 0.905
LOCAL_SITE_SYMBOL_SIZE = 64

# -----------------------------------------------------------------------------
# 0.5-degree statistics export settings
# -----------------------------------------------------------------------------
STATS_RES_DEG = 0.5
STATS_EXPORT_ONLY_VALID_CELLS = True

# -----------------------------------------------------------------------------
# Local panel layout and geographic controls
# -----------------------------------------------------------------------------
EXPAND_LOCAL_EXTENTS_TO_COMMON_RATIO = True
LOCAL_PANEL_PAD_FRAC = 0.035
LOCAL_TARGET_RATIO_SCALE = 1.00
LOCAL_MIN_COSLAT = 0.18
LOCAL_PANEL_TARGET_RATIO = 1.45 * LOCAL_TARGET_RATIO_SCALE

SHOW_REQUESTED_LOCAL_BOX = False
REQUESTED_LOCAL_BOX_COLOR = "black"
REQUESTED_LOCAL_BOX_LINEWIDTH = 0.8
REQUESTED_LOCAL_BOX_LINESTYLE = "--"

# -----------------------------------------------------------------------------
# Colorbar position inside global panel
# -----------------------------------------------------------------------------
COLORBAR_INSET = [0.34, 0.085, 0.32, 0.035]
COLORBAR_TICKS = [0.6, 0.7, 0.8, 0.9, 1.0]

# -----------------------------------------------------------------------------
# Tick and scale-bar settings
# -----------------------------------------------------------------------------
ALL_LOCAL_PANELS_USE_SCALEBAR_ONLY = True
SHOW_TICKS_ON_EVERY_LOCAL_PANEL = False
LOCAL_NUM_XTICKS = 2
LOCAL_NUM_YTICKS = 2

SCALEBAR_TARGET_FRACTION = 0.24
SCALEBAR_XPAD_FRAC = 0.08
SCALEBAR_Y_BAR_FRAC = 0.095
SCALEBAR_BAR_HEIGHT_FRAC = 0.016
SCALEBAR_LABEL_GAP_FRAC = 0.020
SCALEBAR_LABEL_BOX_FACE = "#7f7f7f"
SCALEBAR_LABEL_BOX_ALPHA = 0.92
SCALEBAR_LABEL_BOX_PAD = 0.20

# -----------------------------------------------------------------------------
# Standalone supplementary river-sediment-flux figure settings
# -----------------------------------------------------------------------------
BINS_FLUX = [1, 100, 1e3, 1e4, 1e5, 1e6, 1e7, 1e9]
LABELS_FLUX = ["1-100", "100-1k", "1k-10k", "10k-100k", "100k-1M", "1M-10M", "10M-1B"]
PANEL_C_MARKER_SIZE = 11.0
PANEL_C_MARKER_FACE_COLOR = "#2b6cb0"
PANEL_C_MARKER_EDGE_COLOR = "black"
PANEL_C_ERROR_COLOR = "#b22222"
PANEL_C_ERROR_LINEWIDTH = 1.25
PANEL_C_CAPSIZE = 4
PANEL_C_GRID = True
PANEL_C_GRID_LINESTYLE = "--"
PANEL_C_GRID_LINEWIDTH = 0.45
PANEL_C_GRID_ALPHA = 0.55
PANEL_C_XLABEL = "Mean River sediment flux (tons month$^{-1}$)"
PANEL_C_YLABEL = "Mean WSCI"
PANEL_C_YLIM = None
PANEL_C_XLIM = None

# -----------------------------------------------------------------------------
# Geographic settings
# -----------------------------------------------------------------------------
GLOBAL_EXTENT = (-180.0, 180.0, -90.0, 90.0)
NATURAL_EARTH_LAND_NAME = "land"
NATURAL_EARTH_RIVER_NAME = "rivers_lake_centerlines"

# -----------------------------------------------------------------------------
# Site definitions
# extent = (lonMin, lonMax, latMin, latMax)
#
# Ordering rule requested by the user:
#   Louisiana remains site 1.
#   All remaining sites are ordered from west to east.
#
# The three new windows are centered on the requested locations:
#   Yukon River Delta:              -164.7397 E,  63.3020 N
#   Rufiji Delta-Mafia Channel:       39.5117 E,  -7.7348 N
#   Southern Gulf of Carpentaria:    140.5213 E, -17.4719 N
# -----------------------------------------------------------------------------
SITE_MARKERS = ["o", "s", "^", "D", "P", "X", "v", "<", ">"]

SITES = [
    {
        "id": 1,
        "name": "Louisiana",
        "short_name": "Louisiana",
        "extent": (-94.0, -89.0, 28.5, 30.5),
        "marker": SITE_MARKERS[0],
        "river_source": "louisiana_custom",
    },
    {
        "id": 2,
        "name": "Yukon River Delta",
        "short_name": "Yukon River Delta",
        "extent": (-166.6, -162.9, 62.7, 63.9),
        "marker": SITE_MARKERS[1],
        "river_source": "grit",
    },
    {
        "id": 3,
        "name": "Rio de la Plata estuary",
        "short_name": "Rio de la Plata estuary",
        "extent": (-59.5, -53.0, -36.8, -33.5),
        "marker": SITE_MARKERS[2],
        "river_source": "grit",
    },
    {
        "id": 4,
        "name": "Severn Estuary",
        "short_name": "Severn Estuary",
        "extent": (-3.8, -2.0, 51.0, 51.9),
        "marker": SITE_MARKERS[3],
        "river_source": "grit",
    },
    {
        "id": 5,
        "name": "Nile Delta",
        "short_name": "Nile Delta",
        "extent": (29.0, 33.2, 30.2, 32.3),
        "marker": SITE_MARKERS[4],
        "river_source": "grit",
    },
    {
        "id": 6,
        "name": "Rufiji River Delta-Mafia Channel",
        "short_name": "Rufiji Delta-Mafia Channel",
        "extent": (38.35, 40.65, -8.55, -6.95),
        "marker": SITE_MARKERS[5],
        "river_source": "grit",
    },
    {
        "id": 7,
        "name": "Ganges-Brahmaputra-Meghna Delta",
        "short_name": "Ganges-Brahmaputra Delta",
        "extent": (87.5, 92.8, 20.5, 23.7),
        "marker": SITE_MARKERS[6],
        "river_source": "grit",
    },
    {
        "id": 8,
        "name": "Yellow River Delta",
        "short_name": "Yellow River Delta",
        "extent": (117.8, 120.8, 36.5, 38.4),
        "marker": SITE_MARKERS[7],
        "river_source": "grit",
    },
    {
        "id": 9,
        "name": "Southern Gulf of Carpentaria",
        "short_name": "Gulf of Carpentaria",
        "extent": (139.35, 141.70, -18.20, -16.70),
        "marker": SITE_MARKERS[8],
        "river_source": "grit",
    },
]

# =============================================================================
# Robinson projection constants and caches
# =============================================================================

ROBINSON_X = np.array([
    1.0000, 0.9986, 0.9954, 0.9900, 0.9822,
    0.9730, 0.9600, 0.9427, 0.9216, 0.8962,
    0.8679, 0.8350, 0.7986, 0.7597, 0.7186,
    0.6732, 0.6213, 0.5722, 0.5322
], dtype=np.float64)

ROBINSON_Y = np.array([
    0.0000, 0.0620, 0.1240, 0.1860, 0.2480,
    0.3100, 0.3720, 0.4340, 0.4958, 0.5571,
    0.6176, 0.6769, 0.7346, 0.7903, 0.8435,
    0.8936, 0.9394, 0.9761, 1.0000
], dtype=np.float64)

_PADDED_EXTENT_CACHE = {}
_NATURAL_EARTH_CACHE = {}
_ROBINSON_EDGE_CACHE = {}
_GPKG_LAYER_CACHE = {}
_GRIT_BBOX_CACHE = {}

# =============================================================================
# Basic helpers
# =============================================================================

def check_input_files() -> None:
    """Check that required input files exist."""
    missing = []
    for path in [INPUT_TIF, CSV_FILE_PATH]:
        if not os.path.exists(path):
            missing.append(path)
    if missing:
        raise FileNotFoundError("Missing required input file(s):\n" + "\n".join(missing))

    if REQUIRE_GRIT_LOCAL_RIVERS and not os.path.exists(GRIT_RIVER_GPKG_PATH):
        raise FileNotFoundError(
            "GRIT local-river GeoPackage was not found:\n" + GRIT_RIVER_GPKG_PATH
        )

    if not os.path.exists(LOUISIANA_RIVER_SHP_PATH):
        print(f"[WARNING] Louisiana river shapefile not found: {LOUISIANA_RIVER_SHP_PATH}")
        print("          Louisiana panel will fall back to GRIT rivers.")


def extent_center(extent: Tuple[float, float, float, float]) -> Tuple[float, float]:
    """Return center lon/lat from extent = (xmin, xmax, ymin, ymax)."""
    xmin, xmax, ymin, ymax = extent
    return 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)


def extent_intersects(a: Tuple[float, float, float, float], b: Tuple[float, float, float, float]) -> bool:
    """Return True if two extents intersect. Extent order: xmin, xmax, ymin, ymax."""
    ax0, ax1, ay0, ay1 = a
    bx0, bx1, by0, by1 = b
    return (ax0 <= bx1) and (ax1 >= bx0) and (ay0 <= by1) and (ay1 >= by0)


def geom_bounds_as_extent(geom) -> Optional[Tuple[float, float, float, float]]:
    """Return shapely bounds in extent order, or None for empty geometries."""
    if geom is None or geom.is_empty:
        return None
    xmin, ymin, xmax, ymax = geom.bounds
    return (xmin, xmax, ymin, ymax)


def safe_coslat(lat_deg: float) -> float:
    """Return cos(latitude) with a lower bound to avoid instability at high latitudes."""
    return max(float(np.cos(np.deg2rad(lat_deg))), LOCAL_MIN_COSLAT)


def extent_from_transform(transform, height: int, width: int) -> Tuple[float, float, float, float]:
    """Return lon/lat extent from an affine transform and array shape."""
    corners = [
        transform * (0, 0),
        transform * (width, 0),
        transform * (width, height),
        transform * (0, height),
    ]
    xs = [pt[0] for pt in corners]
    ys = [pt[1] for pt in corners]
    return (float(min(xs)), float(max(xs)), float(min(ys)), float(max(ys)))


def get_padded_local_extent(
    extent: Tuple[float, float, float, float],
    pad_frac: float = LOCAL_PANEL_PAD_FRAC,
    target_ratio: float = LOCAL_PANEL_TARGET_RATIO,
) -> Tuple[float, float, float, float]:
    """Expand a local extent to a common approximate metric aspect ratio."""
    if not EXPAND_LOCAL_EXTENTS_TO_COMMON_RATIO:
        return extent

    key = (tuple(np.round(extent, 8)), float(pad_frac), float(target_ratio))
    if key in _PADDED_EXTENT_CACHE:
        return _PADDED_EXTENT_CACHE[key]

    xmin, xmax, ymin, ymax = extent
    lon_c, lat_c = extent_center(extent)
    coslat = safe_coslat(lat_c)

    lon_span = max(xmax - xmin, 1e-9) * (1.0 + 2.0 * pad_frac)
    lat_span = max(ymax - ymin, 1e-9) * (1.0 + 2.0 * pad_frac)

    x_metric = lon_span * coslat
    y_metric = lat_span
    current_ratio = x_metric / y_metric

    if current_ratio < target_ratio:
        x_metric = y_metric * target_ratio
        lon_span = x_metric / coslat
    else:
        y_metric = x_metric / target_ratio
        lat_span = y_metric

    xmin2 = lon_c - 0.5 * lon_span
    xmax2 = lon_c + 0.5 * lon_span
    ymin2 = max(lat_c - 0.5 * lat_span, -89.9)
    ymax2 = min(lat_c + 0.5 * lat_span, 89.9)

    out = (xmin2, xmax2, ymin2, ymax2)
    _PADDED_EXTENT_CACHE[key] = out
    return out


def expand_extent_by_fraction(
    extent: Tuple[float, float, float, float],
    expand_fraction: float,
) -> Tuple[float, float, float, float]:
    """Expand a lon/lat extent around its center by a fractional amount."""
    xmin, xmax, ymin, ymax = extent
    lon_c, lat_c = extent_center(extent)
    lon_half = 0.5 * max(xmax - xmin, 1e-9) * (1.0 + float(expand_fraction))
    lat_half = 0.5 * max(ymax - ymin, 1e-9) * (1.0 + float(expand_fraction))
    return (
        max(lon_c - lon_half, -180.0),
        min(lon_c + lon_half, 180.0),
        max(lat_c - lat_half, -89.9),
        min(lat_c + lat_half, 89.9),
    )


def get_global_box_extent(site: dict) -> Tuple[float, float, float, float]:
    """Return the manually controlled extent used for the panel-a zoom box."""
    site_id = int(site["id"])
    override = GLOBAL_BOX_EXTENT_OVERRIDES.get(site_id, None)
    if override is not None:
        return tuple(override)
    return expand_extent_by_fraction(site["extent"], GLOBAL_BOX_EXPAND_FRACTION)


def normalize_raster_values(arr: np.ndarray) -> np.ndarray:
    """Clean the science raster and optionally mask values outside the displayed color range."""
    out = np.asarray(arr, dtype=np.float32).copy()
    out[~np.isfinite(out)] = np.nan
    if MASK_VALUES_OUTSIDE_COLOR_RANGE:
        out[(out < DATA_VMIN) | (out > DATA_VMAX)] = np.nan
    return out


def get_data_cmap():
    """Return the already-built WSCI colormap object."""
    return DATA_CMAP

# =============================================================================
# Raster reading helpers
# =============================================================================

def read_native_window(
    path: str,
    requested_extent: Tuple[float, float, float, float],
) -> Tuple[np.ndarray, Tuple[float, float, float, float]]:
    """
    Read a raster window at native resolution.

    Returns
    -------
    arr : np.ndarray
        Native-resolution float32 raster window.
    actual_extent : tuple
        Extent of the returned window in lon/lat order: xmin, xmax, ymin, ymax.
    """
    xmin, xmax, ymin, ymax = requested_extent
    with rasterio.open(path) as src:
        window = window_from_bounds(xmin, ymin, xmax, ymax, transform=src.transform)
        window = window.round_offsets().round_lengths()
        arr = src.read(
            1,
            window=window,
            boundless=True,
            fill_value=np.nan,
            out_dtype="float32",
            resampling=Resampling.nearest,
            masked=False,
        )
        nodata = src.nodata
        if nodata is not None and np.isfinite(nodata):
            arr[arr == nodata] = np.nan

        transform = src.window_transform(window)
        actual_extent = extent_from_transform(transform, arr.shape[0], arr.shape[1])

    return normalize_raster_values(arr), actual_extent

# =============================================================================
# Vector helpers
# =============================================================================

def load_vector_geometries(path: str) -> List:
    """Load geometries from a vector file. GeoPandas is preferred for reliable CRS handling."""
    if path is None or not os.path.exists(path):
        return []

    try:
        import geopandas as gpd
        gdf = gpd.read_file(path)
        if gdf.crs is not None:
            gdf = gdf.to_crs(epsg=4326)
        return [geom for geom in gdf.geometry.values if geom is not None and not geom.is_empty]
    except Exception as exc_gpd:
        try:
            from cartopy.io import shapereader
            reader = shapereader.Reader(path)
            return [geom for geom in reader.geometries() if geom is not None and not geom.is_empty]
        except Exception as exc_cartopy:
            print(f"[WARNING] Could not read vector file: {path}")
            print(f"          GeoPandas error: {exc_gpd}")
            print(f"          Cartopy error  : {exc_cartopy}")
            return []


def load_natural_earth_geometries(scale: str, category: str, name: str) -> List:
    """Load and cache Natural Earth geometries via Cartopy."""
    key = (scale, category, name)
    if key in _NATURAL_EARTH_CACHE:
        return _NATURAL_EARTH_CACHE[key]

    try:
        from cartopy.io import shapereader
        shp_path = shapereader.natural_earth(resolution=scale, category=category, name=name)
        geoms = load_vector_geometries(shp_path)
    except Exception as exc:
        print(f"[WARNING] Could not load Natural Earth {scale}/{category}/{name}: {exc}")
        geoms = []

    _NATURAL_EARTH_CACHE[key] = geoms
    return geoms



def detect_gpkg_layer(path: str, preferred_layer: Optional[str] = None) -> Optional[str]:
    """Return a GeoPackage layer name, preferring a requested or river-like layer."""
    if preferred_layer:
        return preferred_layer

    if path in _GPKG_LAYER_CACHE:
        return _GPKG_LAYER_CACHE[path]

    layer = None
    try:
        import fiona
        layers = list(fiona.listlayers(path))
        if len(layers) == 1:
            layer = layers[0]
        elif len(layers) > 1:
            # Prefer a layer whose name looks like the GRIT segment layer.
            candidates = [
                x for x in layers
                if ("segment" in x.lower()) or ("grit" in x.lower()) or ("river" in x.lower())
            ]
            layer = candidates[0] if candidates else layers[0]
            print(f"[INFO] GeoPackage contains multiple layers: {layers}")
            print(f"       Using layer: {layer}")
    except Exception as exc:
        print(f"[WARNING] Could not list GeoPackage layers for {path}: {exc}")
        layer = None

    _GPKG_LAYER_CACHE[path] = layer
    return layer


def pad_extent_fraction(
    extent: Tuple[float, float, float, float],
    frac: float,
) -> Tuple[float, float, float, float]:
    """Pad a lon/lat extent by a small fraction on all sides."""
    xmin, xmax, ymin, ymax = extent
    dx = max(xmax - xmin, 1e-9) * float(frac)
    dy = max(ymax - ymin, 1e-9) * float(frac)
    return (
        max(-180.0, xmin - dx),
        min(180.0, xmax + dx),
        max(-90.0, ymin - dy),
        min(90.0, ymax + dy),
    )


def load_vector_geometries_bbox(
    path: str,
    extent: Tuple[float, float, float, float],
    layer: Optional[str] = None,
) -> List:
    """
    Read only vector features intersecting a lon/lat bbox.

    The GRIT file supplied for this figure is explicitly EPSG:4326, so the
    panel bbox can be passed directly to Fiona/GeoPandas. Fiona is attempted
    first because its bbox filter is lightweight and works efficiently with
    the GeoPackage spatial index. GeoPandas is used as a fallback.
    """
    if path is None or not os.path.exists(path):
        return []

    xmin, xmax, ymin, ymax = extent
    bbox = (xmin, ymin, xmax, ymax)  # Fiona/GeoPandas bbox order
    cache_key = (
        path,
        layer,
        round(xmin, 6), round(xmax, 6), round(ymin, 6), round(ymax, 6),
    )
    if cache_key in _GRIT_BBOX_CACHE:
        return _GRIT_BBOX_CACHE[cache_key]

    # Preferred route: Fiona bbox filter. This does not load the global file.
    try:
        import fiona
        from shapely.geometry import shape as shapely_shape
        from pyproj import CRS

        open_kwargs = {}
        if layer is not None:
            open_kwargs["layer"] = layer

        with fiona.open(path, **open_kwargs) as src:
            src_crs = None
            try:
                if src.crs_wkt:
                    src_crs = CRS.from_wkt(src.crs_wkt)
                elif src.crs:
                    src_crs = CRS.from_user_input(src.crs)
            except Exception:
                src_crs = None

            if src_crs is not None and src_crs.to_epsg() not in (None, 4326):
                raise ValueError(
                    f"Expected EPSG:4326 GRIT data but found {src_crs.to_string()}"
                )

            geoms = []
            for feat in src.filter(bbox=bbox):
                geom_mapping = feat.get("geometry")
                if geom_mapping is None:
                    continue
                geom = shapely_shape(geom_mapping)
                if geom is not None and not geom.is_empty:
                    geoms.append(geom)

        geoms = filter_geometries_by_extent(geoms, extent)
        _GRIT_BBOX_CACHE[cache_key] = geoms
        return geoms

    except Exception as exc_fiona:
        # Fallback: GeoPandas can also pass a bbox spatial filter to GDAL.
        try:
            import geopandas as gpd

            read_kwargs = {"bbox": bbox}
            if layer is not None:
                read_kwargs["layer"] = layer
            gdf = gpd.read_file(path, **read_kwargs)
            if gdf.crs is not None:
                gdf = gdf.to_crs(epsg=4326)
            geoms = [
                geom for geom in gdf.geometry.values
                if geom is not None and not geom.is_empty
            ]
            geoms = filter_geometries_by_extent(geoms, extent)
            _GRIT_BBOX_CACHE[cache_key] = geoms
            return geoms

        except Exception as exc_gpd:
            print(f"[WARNING] Could not bbox-read vector file: {path}")
            print(f"          Fiona error    : {exc_fiona}")
            print(f"          GeoPandas error: {exc_gpd}")
            _GRIT_BBOX_CACHE[cache_key] = []
            return []


def load_grit_rivers_for_extent(
    extent: Tuple[float, float, float, float],
) -> List:
    """Read GRIT river geometries intersecting one local-panel extent."""
    query_extent = pad_extent_fraction(extent, GRIT_BBOX_PAD_FRACTION)
    layer = detect_gpkg_layer(GRIT_RIVER_GPKG_PATH, GRIT_GPKG_LAYER)
    return load_vector_geometries_bbox(
        GRIT_RIVER_GPKG_PATH,
        query_extent,
        layer=layer,
    )


def filter_geometries_by_extent(geoms: Sequence, extent: Tuple[float, float, float, float]) -> List:
    """Filter geometries by simple bounding-box intersection."""
    out = []
    for geom in geoms:
        gb = geom_bounds_as_extent(geom)
        if gb is not None and extent_intersects(gb, extent):
            out.append(geom)
    return out


def rasterize_geometry_mask(
    geoms: Sequence,
    extent: Tuple[float, float, float, float],
    shape: Tuple[int, int],
    all_touched: bool,
) -> np.ndarray:
    """Rasterize arbitrary geometries to the target grid."""
    height, width = shape
    if len(geoms) == 0 or height <= 0 or width <= 0:
        return np.zeros((height, width), dtype=np.uint8)

    local_geoms = filter_geometries_by_extent(geoms, extent)
    if len(local_geoms) == 0:
        return np.zeros((height, width), dtype=np.uint8)

    transform = from_bounds(extent[0], extent[2], extent[1], extent[3], width, height)
    mask = rasterize(
        [(geom, 1) for geom in local_geoms],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        all_touched=all_touched,
        dtype="uint8",
    )
    return mask


def rasterize_land_mask(
    land_geoms: Sequence,
    extent: Tuple[float, float, float, float],
    shape: Tuple[int, int],
    all_touched: bool,
) -> np.ndarray:
    """Rasterize land polygons to the target grid."""
    return rasterize_geometry_mask(land_geoms, extent, shape, all_touched)


def mask_data_on_land(data: np.ndarray, land_mask: np.ndarray) -> np.ndarray:
    """Hide data pixels where land_mask == 1."""
    out = np.array(data, copy=True, dtype=np.float32)
    if land_mask.shape == out.shape:
        out[land_mask == 1] = np.nan
    return out


def iter_line_parts(geom) -> Iterable[np.ndarray]:
    """Yield Nx2 lon/lat arrays from LineString/MultiLineString geometries."""
    if geom is None or geom.is_empty:
        return

    gtype = geom.geom_type
    if gtype == "LineString":
        coords = np.asarray(geom.coords, dtype=float)
        if coords.shape[0] >= 2:
            yield coords[:, :2]
    elif gtype == "MultiLineString":
        for part in geom.geoms:
            yield from iter_line_parts(part)
    elif gtype == "GeometryCollection":
        for part in geom.geoms:
            yield from iter_line_parts(part)


def plot_local_rivers(
    ax,
    geoms: Sequence,
    extent: Tuple[float, float, float, float],
    linewidth: float = LOCAL_RIVER_LINEWIDTH,
    alpha: float = LOCAL_RIVER_ALPHA,
) -> int:
    """
    Plot river line geometries in lon/lat coordinates on a local panel.

    Returns
    -------
    int
        Number of individual line parts drawn.
    """
    local_geoms = filter_geometries_by_extent(geoms, extent)
    segments = []
    for geom in local_geoms:
        for coords in iter_line_parts(geom):
            if coords.shape[0] >= 2:
                segments.append(coords[:, :2])

    if len(segments) == 0:
        return 0

    if USE_LINECOLLECTION_FOR_LOCAL_RIVERS:
        collection = LineCollection(
            segments,
            colors=RIVER_COLOR,
            linewidths=linewidth,
            alpha=alpha,
            zorder=12,
        )
        try:
            collection.set_capstyle("round")
            collection.set_joinstyle("round")
        except Exception:
            pass
        ax.add_collection(collection)
    else:
        for coords in segments:
            ax.plot(
                coords[:, 0], coords[:, 1],
                color=RIVER_COLOR,
                linewidth=linewidth,
                alpha=alpha,
                solid_capstyle="round",
                zorder=12,
            )

    return len(segments)


def plot_global_rivers_robinson(
    ax,
    geoms: Sequence,
    linewidth: float = GLOBAL_RIVER_LINEWIDTH,
    alpha: float = GLOBAL_RIVER_ALPHA,
) -> None:
    """Project and plot global river line geometries in Robinson coordinates."""
    for geom in geoms:
        for coords in iter_line_parts(geom):
            x, y = robinson_project(coords[:, 0], coords[:, 1])
            ax.plot(
                x,
                y,
                color=RIVER_COLOR,
                linewidth=linewidth,
                alpha=alpha,
                solid_capstyle="round",
                zorder=6,
            )

# =============================================================================
# Robinson projection helpers
# =============================================================================

def robinson_project(lon_deg, lat_deg):
    """Manual Robinson projection, avoiding PROJ dependence for the global panel."""
    lon = np.asarray(lon_deg, dtype=np.float64)
    lat = np.asarray(lat_deg, dtype=np.float64)

    abs_lat = np.clip(np.abs(lat), 0.0, 90.0)
    idx = np.floor(abs_lat / 5.0).astype(int)
    idx = np.clip(idx, 0, 17)
    frac = (abs_lat - idx * 5.0) / 5.0

    xcoef = ROBINSON_X[idx] * (1.0 - frac) + ROBINSON_X[idx + 1] * frac
    ycoef = ROBINSON_Y[idx] * (1.0 - frac) + ROBINSON_Y[idx + 1] * frac

    x = 0.8487 * xcoef * np.deg2rad(lon)
    y = 1.3523 * np.sign(lat) * ycoef
    return x.astype(np.float32), y.astype(np.float32)


def get_robinson_edges_for_global(width: int, height: int):
    """Build and cache the Robinson edge mesh for the full global domain."""
    key = (width, height)
    if key in _ROBINSON_EDGE_CACHE:
        return _ROBINSON_EDGE_CACHE[key]

    lon_edges = np.linspace(-180.0, 180.0, width + 1, dtype=np.float32)
    lat_edges = np.linspace(90.0, -90.0, height + 1, dtype=np.float32)
    lon2d, lat2d = np.meshgrid(lon_edges, lat_edges)
    x_edges, y_edges = robinson_project(lon2d, lat2d)
    xlim = (float(np.nanmin(x_edges)), float(np.nanmax(x_edges)))
    ylim = (float(np.nanmin(y_edges)), float(np.nanmax(y_edges)))

    _ROBINSON_EDGE_CACHE[key] = (x_edges, y_edges, xlim, ylim)
    return _ROBINSON_EDGE_CACHE[key]


def build_projected_extent_outline(extent: Tuple[float, float, float, float], n: int = 120):
    """Build a curved Robinson outline for a lon/lat rectangle."""
    xmin, xmax, ymin, ymax = extent
    lon_top = np.linspace(xmin, xmax, n)
    lat_top = np.full_like(lon_top, ymax)
    lon_right = np.full(n, xmax)
    lat_right = np.linspace(ymax, ymin, n)
    lon_bottom = np.linspace(xmax, xmin, n)
    lat_bottom = np.full_like(lon_bottom, ymin)
    lon_left = np.full(n, xmin)
    lat_left = np.linspace(ymin, ymax, n)

    lon = np.concatenate([lon_top, lon_right, lon_bottom, lon_left])
    lat = np.concatenate([lat_top, lat_right, lat_bottom, lat_left])
    return robinson_project(lon, lat)

# =============================================================================
# Global plotting helpers
# =============================================================================

def draw_global_land_fill(ax, land_mask: np.ndarray) -> None:
    """Draw a light-gray land fill on the global Robinson panel."""
    if land_mask is None or land_mask.size == 0:
        return
    h, w = land_mask.shape
    x_edges, y_edges, xlim, ylim = get_robinson_edges_for_global(w, h)
    land_float = np.where(land_mask == 1, 1.0, np.nan).astype(np.float32)
    ax.pcolormesh(
        x_edges,
        y_edges,
        np.ma.masked_invalid(land_float),
        shading="auto",
        cmap=ListedColormap([LAND_COLOR]),
        vmin=0,
        vmax=1,
        rasterized=True,
        zorder=4,
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")


def add_global_land_boundary(ax, land_mask: np.ndarray) -> None:
    """Draw a thin coastline/land boundary on the global Robinson panel."""
    if land_mask is None or land_mask.size == 0:
        return

    yy = np.linspace(90.0, -90.0, land_mask.shape[0], dtype=np.float32)
    xx = np.linspace(-180.0, 180.0, land_mask.shape[1], dtype=np.float32)
    lon2d, lat2d = np.meshgrid(xx, yy)
    x2d, y2d = robinson_project(lon2d, lat2d)

    try:
        ax.contour(
            x2d,
            y2d,
            land_mask.astype(float),
            levels=[0.5],
            colors=[LAND_EDGE_COLOR],
            linewidths=LAND_EDGE_LINEWIDTH_GLOBAL,
            zorder=5,
        )
    except Exception:
        pass


def update_histogram_counts(hist_counts: np.ndarray, values: np.ndarray, hist_edges: np.ndarray) -> np.ndarray:
    """Update histogram counts using finite values within the requested WSCI range."""
    vals = values[np.isfinite(values)]
    if vals.size == 0:
        return hist_counts
    lo, hi = hist_edges[0], hist_edges[-1]
    vals = vals[(vals >= lo) & (vals <= hi)]
    if vals.size == 0:
        return hist_counts
    hist_counts += np.histogram(vals, bins=hist_edges)[0].astype(np.int64)
    return hist_counts


def plot_global_native_scatter_robinson(
    ax,
    raster_path: str,
    hist_edges: np.ndarray,
) -> Tuple[int, np.ndarray]:
    """
    Plot the global raster at native resolution as a block-wise scatter layer.

    Returns
    -------
    n_total : int
        Number of valid points plotted.
    hist_counts : np.ndarray
        Histogram counts for WSCI inset, using the original valid WSCI raster values.
    """
    n_total = 0
    hist_counts = np.zeros(len(hist_edges) - 1, dtype=np.int64)
    data_cmap = get_data_cmap()

    with rasterio.open(raster_path) as src:
        nodata = src.nodata
        step = max(int(GLOBAL_SCATTER_PIXEL_STEP), 1)

        for _, window in src.block_windows(1):
            arr = src.read(1, window=window, masked=False, out_dtype="float32")
            if nodata is not None and np.isfinite(nodata):
                arr[arr == nodata] = np.nan

            arr = normalize_raster_values(arr)
            transform = src.window_transform(window)

            # Apply the manuscript analysis latitude domain before either
            # updating the histogram/CDF inset or plotting the global data.
            lat_rows = transform.f + (
                np.arange(window.height, dtype=np.float64) + 0.5
            ) * transform.e
            keep_rows = lat_rows <= MAX_ANALYSIS_LAT
            if not np.all(keep_rows):
                arr[~keep_rows, :] = np.nan

            hist_counts = update_histogram_counts(hist_counts, arr, hist_edges)

            if step > 1:
                arr = arr[::step, ::step]

            valid = np.isfinite(arr)
            if not np.any(valid):
                continue

            xs = transform.c + (np.arange(window.width, dtype=np.float64)[::step] + 0.5) * transform.a
            ys = transform.f + (np.arange(window.height, dtype=np.float64)[::step] + 0.5) * transform.e
            lon2d, lat2d = np.meshgrid(xs, ys)

            vals = arr[valid]
            lons = lon2d[valid]
            lats = lat2d[valid]
            x, y = robinson_project(lons, lats)

            ax.scatter(
                x,
                y,
                c=vals,
                cmap=data_cmap,
                norm=DATA_NORM,
                s=GLOBAL_SCATTER_SIZE,
                marker=GLOBAL_SCATTER_MARKER,
                linewidths=0.0,
                edgecolors="none",
                alpha=GLOBAL_SCATTER_ALPHA,
                rasterized=True,
                zorder=GLOBAL_SCATTER_ZORDER,
            )
            n_total += int(vals.size)

    return n_total, hist_counts


def decorate_global_panel(ax, sites: Sequence[dict]) -> None:
    """Add manually adjustable site boxes and numeric labels to the global panel."""
    if SHOW_GLOBAL_ZOOM_BOXES:
        for site in sites:
            box_extent = get_global_box_extent(site)
            x_box, y_box = build_projected_extent_outline(box_extent, n=90)
            ax.plot(
                x_box,
                y_box,
                color=GLOBAL_BOX_COLOR,
                linewidth=GLOBAL_BOX_LINEWIDTH,
                linestyle=GLOBAL_BOX_LINESTYLE,
                zorder=8,
            )

    if SHOW_GLOBAL_MARKERS:
        for site in sites:
            site_id = int(site["id"])
            lon_c, lat_c = extent_center(site["extent"])

            # Keep the small marker at the true site center.
            x, y = robinson_project(lon_c, lat_c)
            ax.scatter(
                x,
                y,
                s=GLOBAL_MARKER_SIZE,
                marker=site["marker"],
                facecolor=GLOBAL_MARKER_FACE,
                edgecolor=GLOBAL_MARKER_EDGE,
                linewidth=GLOBAL_MARKER_LINEWIDTH,
                zorder=9,
            )

            # Move the numeric label independently from the marker.
            dlon, dlat = GLOBAL_SITE_LABEL_LONLAT_OFFSETS_DEG.get(site_id, (0.0, 0.0))
            label_x, label_y = robinson_project(lon_c + dlon, lat_c + dlat)
            dx_proj, dy_proj = GLOBAL_SITE_LABEL_PROJECTED_OFFSETS.get(
                site_id,
                (GLOBAL_MARKER_LABEL_DX, GLOBAL_MARKER_LABEL_DY),
            )
            txt = ax.text(
                float(label_x) + dx_proj,
                float(label_y) + dy_proj,
                str(site_id),
                fontsize=GLOBAL_MARKER_LABEL_FONT_SIZE,
                fontweight="bold",
                color="black",
                ha="center",
                va="center",
                zorder=10,
                bbox=dict(
                    boxstyle=f"round,pad={GLOBAL_SITE_LABEL_BOX_PAD}",
                    facecolor=GLOBAL_SITE_LABEL_BOX_FACE,
                    edgecolor=GLOBAL_SITE_LABEL_BOX_EDGE,
                    linewidth=GLOBAL_SITE_LABEL_BOX_LINEWIDTH,
                    alpha=GLOBAL_SITE_LABEL_BOX_ALPHA,
                ),
            )
            txt.set_path_effects([pe.withStroke(linewidth=1.1, foreground="white")])


def add_wsci_histogram_cdf_inset(ax_global, hist_counts: np.ndarray, hist_edges: np.ndarray) -> None:
    """Add WSCI histogram and cumulative-frequency inset to panel a."""
    if not SHOW_WSCI_INSET:
        return

    total = hist_counts.sum()
    if total <= 0:
        print("[WARNING] No valid WSCI values available for inset histogram.")
        return

    centers = 0.5 * (hist_edges[:-1] + hist_edges[1:])
    widths = np.diff(hist_edges)
    heights = hist_counts.astype(float)
    if WSCI_HIST_USE_PERCENT:
        heights = heights / total * 100.0
    cdf = np.cumsum(hist_counts.astype(float)) / total * 100.0

    data_cmap = get_data_cmap()
    below_threshold_color = (
        data_cmap(DATA_NORM(DATA_VMIN))
        if WSCI_HIST_BELOW_THRESHOLD_COLOR is None
        else WSCI_HIST_BELOW_THRESHOLD_COLOR
    )
    colors = []
    for x in centers:
        if x < WSCI_HIST_THRESHOLD:
            colors.append(below_threshold_color)
        else:
            colors.append(data_cmap(DATA_NORM(x)))

    ax_hist = ax_global.inset_axes(WSCI_INSET_AXES)
    ax_hist.set_facecolor(WSCI_INSET_FACE_COLOR)
    ax_hist.patch.set_alpha(WSCI_INSET_ALPHA)

    ax_hist.bar(
        centers,
        heights,
        width=widths,
        align="center",
        color=colors,
        edgecolor=WSCI_HIST_EDGE_COLOR,
        linewidth=0.0,
        alpha=WSCI_HIST_ALPHA,
        zorder=2,
    )
    ax_hist.axvline(
        WSCI_HIST_THRESHOLD,
        color=WSCI_THRESHOLD_LINE_COLOR,
        linestyle=WSCI_THRESHOLD_LINESTYLE,
        linewidth=WSCI_THRESHOLD_LINEWIDTH,
        zorder=4,
    )

    ax_cdf = ax_hist.twinx()
    ax_cdf.plot(
        centers,
        cdf,
        color=WSCI_CDF_LINE_COLOR,
        linewidth=WSCI_CDF_LINEWIDTH,
        zorder=5,
    )

    ax_hist.set_xlim(WSCI_HIST_RANGE)
    ax_cdf.set_ylim(0, 100)
    ax_hist.set_xticks(WSCI_INSET_XTICKS)
    ax_hist.set_xlabel(WSCI_INSET_XLABEL, fontsize=INSET_AXIS_LABEL_FONT_SIZE, labelpad=1.5)
    ax_hist.set_ylabel(WSCI_INSET_YLABEL, fontsize=INSET_AXIS_LABEL_FONT_SIZE, labelpad=1.5)
    ax_cdf.set_ylabel(WSCI_INSET_CDF_YLABEL, fontsize=INSET_AXIS_LABEL_FONT_SIZE, labelpad=1.5)

    ax_hist.tick_params(axis="both", labelsize=INSET_TICK_FONT_SIZE, length=2.0, width=0.55, pad=1.3)
    ax_cdf.tick_params(axis="y", labelsize=INSET_TICK_FONT_SIZE, length=2.0, width=0.55, pad=1.3)

    if WSCI_INSET_GRID:
        ax_hist.grid(True, linewidth=0.3, alpha=0.35)

    for spine in ax_hist.spines.values():
        spine.set_linewidth(WSCI_INSET_SPINE_LINEWIDTH)
    for spine in ax_cdf.spines.values():
        spine.set_linewidth(WSCI_INSET_SPINE_LINEWIDTH)


# =============================================================================
# Local panel helpers
# =============================================================================

def add_panel_label(ax, label: str, pos: Tuple[float, float]) -> None:
    """Add a panel label such as A, B, or C."""
    txt = ax.text(
        pos[0],
        pos[1],
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=PANEL_LABEL_FONT_SIZE,
        fontweight="bold",
        color="black",
        zorder=60,
    )
    txt.set_path_effects([pe.withStroke(linewidth=2.6, foreground="white")])


def add_local_title(ax, site: dict) -> None:
    """Add the location name to a manually adjustable position in the local panel."""
    if not SHOW_LOCAL_TITLES:
        return

    site_id = int(site["id"])
    title_x, title_y = LOCAL_TITLE_AXES_POS_OVERRIDES.get(
        site_id,
        (LOCAL_TITLE_AXES_X, LOCAL_TITLE_AXES_Y),
    )

    txt = ax.text(
        title_x,
        title_y,
        site["short_name"],
        transform=ax.transAxes,
        ha=LOCAL_TITLE_HA,
        va=LOCAL_TITLE_VA,
        fontsize=LOCAL_TITLE_FONT_SIZE,
        color="black",
        zorder=39,
        bbox=dict(facecolor="white", edgecolor="none", alpha=LOCAL_TITLE_BOX_ALPHA, pad=LOCAL_TITLE_BOX_PAD),
    )
    txt.set_path_effects([pe.withStroke(linewidth=1.2, foreground="white")])


def add_local_site_symbol(ax, site: dict) -> None:
    """Draw the same marker shape used on the global panel inside the local panel."""
    if not SHOW_LOCAL_SITE_SYMBOL:
        return
    ax.scatter(
        LOCAL_SITE_SYMBOL_AXES_X,
        LOCAL_SITE_SYMBOL_AXES_Y,
        s=LOCAL_SITE_SYMBOL_SIZE,
        marker=site["marker"],
        facecolor=GLOBAL_MARKER_FACE,
        edgecolor=GLOBAL_MARKER_EDGE,
        linewidth=GLOBAL_MARKER_LINEWIDTH,
        transform=ax.transAxes,
        zorder=41,
        clip_on=False,
    )



def add_local_site_number(ax, site: dict) -> None:
    """Draw the numeric site id in the upper-left of each local panel."""
    if not SHOW_LOCAL_SITE_NUMBER:
        return

    site_id = int(site["id"])
    x_pos, y_pos = LOCAL_SITE_NUMBER_AXES_POS_OVERRIDES.get(
        site_id,
        (LOCAL_SITE_NUMBER_AXES_X, LOCAL_SITE_NUMBER_AXES_Y),
    )
    txt = ax.text(
        x_pos,
        y_pos,
        str(site_id),
        transform=ax.transAxes,
        ha="center",
        va="center",
        fontsize=LOCAL_SITE_NUMBER_FONT_SIZE,
        fontweight="bold",
        color="black",
        zorder=42,
        clip_on=False,
        bbox=dict(
            boxstyle=f"round,pad={LOCAL_SITE_NUMBER_BOX_PAD}",
            facecolor=LOCAL_SITE_NUMBER_BOX_FACE,
            edgecolor=LOCAL_SITE_NUMBER_BOX_EDGE,
            linewidth=LOCAL_SITE_NUMBER_BOX_LINEWIDTH,
            alpha=LOCAL_SITE_NUMBER_BOX_ALPHA,
        ),
    )
    txt.set_path_effects([pe.withStroke(linewidth=1.0, foreground="white")])


def apply_local_aspect(ax, extent: Tuple[float, float, float, float]) -> None:
    """Apply a latitude-aware aspect ratio for local lon/lat panels."""
    _, lat_c = extent_center(extent)
    ax.set_aspect(1.0 / safe_coslat(lat_c), adjustable="box")


def choose_nice_scalebar_length_km(max_length_km: float) -> float:
    """Choose a simple scale-bar length that fits within the panel."""
    if not np.isfinite(max_length_km) or max_length_km <= 0:
        return 1.0
    candidates = []
    for exponent in range(-1, 6):
        base = 10.0 ** exponent
        for multiplier in (1.0, 2.0, 5.0):
            candidates.append(multiplier * base)
    candidates = np.array(sorted(candidates), dtype=float)
    valid = candidates[candidates <= max_length_km]
    if valid.size == 0:
        return float(candidates[0])
    return float(valid[-1])


def format_scalebar_label(length_km: float) -> str:
    """Format the local scale-bar label."""
    if length_km < 1.0:
        return f"{int(round(length_km * 1000.0))} m"
    if np.isclose(length_km, round(length_km)):
        return f"{int(round(length_km))} km"
    return f"{length_km:.1f} km"


def add_local_scalebar(ax, extent: Tuple[float, float, float, float]) -> None:
    """Add a lower-left scale bar to a local panel."""
    xmin, xmax, ymin, ymax = extent
    _, lat_c = extent_center(extent)
    width_km = max((xmax - xmin) * 111.32 * safe_coslat(lat_c), 1e-6)
    length_km = choose_nice_scalebar_length_km(width_km * SCALEBAR_TARGET_FRACTION)

    deg_per_km_lon = 1.0 / (111.32 * safe_coslat(lat_c))
    bar_length_deg = length_km * deg_per_km_lon

    x_span = xmax - xmin
    y_span = ymax - ymin
    x0 = xmin + SCALEBAR_XPAD_FRAC * x_span
    x1 = x0 + bar_length_deg
    x_max_allowed = xmax - SCALEBAR_XPAD_FRAC * x_span
    if x1 > x_max_allowed:
        x1 = x_max_allowed
        x0 = x1 - bar_length_deg

    bar_y0 = ymin + SCALEBAR_Y_BAR_FRAC * y_span
    bar_height = SCALEBAR_BAR_HEIGHT_FRAC * y_span
    label_y = bar_y0 + bar_height + SCALEBAR_LABEL_GAP_FRAC * y_span

    ax.add_patch(Rectangle(
        (x0, bar_y0),
        x1 - x0,
        bar_height,
        facecolor="black",
        edgecolor="none",
        linewidth=0.0,
        zorder=30,
    ))
    ax.text(
        0.5 * (x0 + x1),
        label_y,
        format_scalebar_label(length_km),
        ha="center",
        va="bottom",
        fontsize=SCALEBAR_FONT_SIZE,
        color="white",
        zorder=31,
        bbox=dict(
            boxstyle=f"round,pad={SCALEBAR_LABEL_BOX_PAD}",
            facecolor=SCALEBAR_LABEL_BOX_FACE,
            edgecolor="none",
            linewidth=0.0,
            alpha=SCALEBAR_LABEL_BOX_ALPHA,
        ),
    )


def degree_label_lon_local(x: float, span: float) -> str:
    """Format longitude tick labels."""
    fmt = "{:.2f}" if span < 1.0 else ("{:.1f}" if span < 5.0 else "{:.0f}")
    if np.isclose(x, 0):
        return "0°"
    return fmt.format(abs(x)) + ("°W" if x < 0 else "°E")


def degree_label_lat_local(y: float, span: float) -> str:
    """Format latitude tick labels."""
    fmt = "{:.2f}" if span < 1.0 else ("{:.1f}" if span < 5.0 else "{:.0f}")
    if np.isclose(y, 0):
        return "0°"
    return fmt.format(abs(y)) + ("°S" if y < 0 else "°N")


def set_local_ticks(ax, extent: Tuple[float, float, float, float], show_x: bool, show_y: bool) -> None:
    """Configure local lon/lat ticks, or hide them for scale-bar-only panels."""
    xmin, xmax, ymin, ymax = extent
    if show_x:
        xticks = np.linspace(xmin, xmax, LOCAL_NUM_XTICKS)
        ax.set_xticks(xticks)
        ax.set_xticklabels([degree_label_lon_local(x, xmax - xmin) for x in xticks])
    else:
        ax.set_xticks([])
        ax.set_xticklabels([])

    if show_y:
        yticks = np.linspace(ymin, ymax, LOCAL_NUM_YTICKS)
        ax.set_yticks(yticks)
        ax.set_yticklabels([degree_label_lat_local(y, ymax - ymin) for y in yticks])
    else:
        ax.set_yticks([])
        ax.set_yticklabels([])

    ax.tick_params(
        axis="both",
        which="major",
        length=2.2 if (show_x or show_y) else 0.0,
        width=0.7,
        pad=1.5,
        bottom=show_x,
        top=False,
        left=show_y,
        right=False,
        labelbottom=show_x,
        labelleft=show_y,
    )


def draw_requested_extent_box(ax, original_extent: Tuple[float, float, float, float]) -> None:
    """Optionally draw the requested local extent inside the padded panel."""
    if not SHOW_REQUESTED_LOCAL_BOX:
        return
    xmin, xmax, ymin, ymax = original_extent
    xs = [xmin, xmax, xmax, xmin, xmin]
    ys = [ymin, ymin, ymax, ymax, ymin]
    ax.plot(
        xs,
        ys,
        color=REQUESTED_LOCAL_BOX_COLOR,
        linewidth=REQUESTED_LOCAL_BOX_LINEWIDTH,
        linestyle=REQUESTED_LOCAL_BOX_LINESTYLE,
        zorder=22,
    )


def draw_land_overlay_local(ax, land_mask: np.ndarray, extent: Tuple[float, float, float, float]) -> None:
    """Draw light-gray land and a coastline contour on a local panel."""
    if land_mask is None or land_mask.size == 0:
        return

    land_float = np.where(land_mask == 1, 1.0, np.nan).astype(np.float32)
    ax.imshow(
        land_float,
        extent=extent,
        origin="upper",
        cmap=ListedColormap([LAND_COLOR]),
        vmin=0,
        vmax=1,
        interpolation="nearest",
        zorder=8,
    )

    try:
        yy = np.linspace(extent[3], extent[2], land_mask.shape[0])
        xx = np.linspace(extent[0], extent[1], land_mask.shape[1])
        ax.contour(
            xx,
            yy,
            land_mask.astype(float),
            levels=[0.5],
            colors=[LAND_EDGE_COLOR],
            linewidths=LAND_EDGE_LINEWIDTH_LOCAL,
            zorder=11,
        )
    except Exception:
        pass

# =============================================================================
# Panel-c data and plotting
# =============================================================================

def compute_flux_binned_statistics(df: pd.DataFrame) -> pd.DataFrame:
    """Compute mean/SEM/count statistics in river sediment-flux bins."""
    required_cols = ["average_sediment_flux_tons_per_month", "mean_correlation"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in CSV: {missing}")

    df = df.copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=required_cols)

    df["flux_bin"] = pd.cut(
        df["average_sediment_flux_tons_per_month"],
        bins=BINS_FLUX,
        labels=LABELS_FLUX,
        right=False,
    )

    df_binned = df.dropna(subset=["flux_bin"])
    grouped = df_binned.groupby("flux_bin", observed=False)

    mean_flux = grouped["average_sediment_flux_tons_per_month"].mean()
    mean_wsci = grouped["mean_correlation"].mean()
    count_flux = grouped.size()
    sem_wsci = grouped["mean_correlation"].sem().fillna(0.0)

    results = pd.DataFrame({
        "flux_bin": mean_flux.index.astype(str),
        "mean_sediment_flux": mean_flux.values,
        "mean_wsci": mean_wsci.values,
        "count": count_flux.values,
        "sem_wsci": sem_wsci.values,
    })
    results = results.replace([np.inf, -np.inf], np.nan)
    return results


def plot_panel_c_river_sediment_flux(ax, flux_stats: pd.DataFrame) -> None:
    """Draw the original Fig.4 panel b as Figure 5 panel c."""
    valid = (
        np.isfinite(flux_stats["mean_sediment_flux"].values) &
        np.isfinite(flux_stats["mean_wsci"].values)
    )
    x = flux_stats.loc[valid, "mean_sediment_flux"].values.astype(float)
    y = flux_stats.loc[valid, "mean_wsci"].values.astype(float)
    yerr = flux_stats.loc[valid, "sem_wsci"].values.astype(float)
    yerr[~np.isfinite(yerr)] = 0.0

    ax.errorbar(
        x,
        y,
        yerr=yerr,
        fmt="o",
        ecolor=PANEL_C_ERROR_COLOR,
        elinewidth=PANEL_C_ERROR_LINEWIDTH,
        capsize=PANEL_C_CAPSIZE,
        capthick=PANEL_C_ERROR_LINEWIDTH,
        markeredgecolor=PANEL_C_MARKER_EDGE_COLOR,
        markerfacecolor=PANEL_C_MARKER_FACE_COLOR,
        markersize=PANEL_C_MARKER_SIZE,
        linestyle="none",
        zorder=4,
    )
    ax.set_xscale("log")
    ax.set_xlabel(PANEL_C_XLABEL, fontsize=PANEL_C_AXIS_LABEL_FONT_SIZE)
    ax.set_ylabel(PANEL_C_YLABEL, fontsize=PANEL_C_AXIS_LABEL_FONT_SIZE)
    ax.tick_params(axis="both", labelsize=PANEL_C_TICK_FONT_SIZE)

    if PANEL_C_XLIM is not None:
        ax.set_xlim(PANEL_C_XLIM)
    if PANEL_C_YLIM is not None:
        ax.set_ylim(PANEL_C_YLIM)

    if PANEL_C_GRID:
        ax.grid(
            True,
            which="both",
            linestyle=PANEL_C_GRID_LINESTYLE,
            linewidth=PANEL_C_GRID_LINEWIDTH,
            alpha=PANEL_C_GRID_ALPHA,
            zorder=1,
        )

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

# =============================================================================
# 0.5-degree statistics export
# =============================================================================

def export_0p5deg_statistics(
    raster_path: str,
    out_csv: str,
    res_deg: float = STATS_RES_DEG,
) -> None:
    """
    Aggregate the native raster to a regular 0.5-degree grid and save one CSV.

    Statistics summarize valid WSCI raster values within the manuscript
    analysis domain (latitude <= MAX_ANALYSIS_LAT).
    """
    nlon = int(round(360.0 / res_deg))
    nlat = int(round(180.0 / res_deg))
    ncell = nlon * nlat

    counts = np.zeros(ncell, dtype=np.int64)
    sums = np.zeros(ncell, dtype=np.float64)
    sums_sq = np.zeros(ncell, dtype=np.float64)
    mins = np.full(ncell, np.inf, dtype=np.float64)
    maxs = np.full(ncell, -np.inf, dtype=np.float64)


    with rasterio.open(raster_path) as src:
        nodata = src.nodata
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window, masked=False, out_dtype="float64")
            if nodata is not None and np.isfinite(nodata):
                arr[arr == nodata] = np.nan
            arr = normalize_raster_values(arr)

            transform = src.window_transform(window)

            xs = transform.c + (np.arange(window.width, dtype=np.float64) + 0.5) * transform.a
            ys = transform.f + (np.arange(window.height, dtype=np.float64) + 0.5) * transform.e
            lon2d, lat2d = np.meshgrid(xs, ys)

            valid = np.isfinite(arr) & (lat2d <= MAX_ANALYSIS_LAT)
            if not np.any(valid):
                continue

            vals = arr[valid]
            lons = lon2d[valid]
            lats = lat2d[valid]

            col_idx = np.floor((lons + 180.0) / res_deg).astype(np.int64)
            row_idx = np.floor((lats + 90.0) / res_deg).astype(np.int64)

            inside = (
                (col_idx >= 0) & (col_idx < nlon) &
                (row_idx >= 0) & (row_idx < nlat)
            )
            if not np.any(inside):
                continue

            col_idx = col_idx[inside]
            row_idx = row_idx[inside]
            vals = vals[inside]
            flat_idx = row_idx * nlon + col_idx

            counts += np.bincount(flat_idx, minlength=ncell)
            sums += np.bincount(flat_idx, weights=vals, minlength=ncell)
            sums_sq += np.bincount(flat_idx, weights=vals * vals, minlength=ncell)
            np.minimum.at(mins, flat_idx, vals)
            np.maximum.at(maxs, flat_idx, vals)

    valid_cells = np.where(counts > 0)[0]
    rows = []
    for idx in valid_cells:
        row_idx = idx // nlon
        col_idx = idx % nlon
        cnt = int(counts[idx])
        mean_val = sums[idx] / cnt
        var_val = max(sums_sq[idx] / cnt - mean_val * mean_val, 0.0)
        std_val = var_val ** 0.5
        lon_center = -180.0 + (col_idx + 0.5) * res_deg
        lat_center = -90.0 + (row_idx + 0.5) * res_deg
        rows.append([
            lon_center,
            lat_center,
            mean_val,
            std_val,
            mins[idx],
            maxs[idx],
            cnt,
        ])

    if len(rows) > 0:
        arr_rows = np.asarray(rows, dtype=np.float64)
        order = np.lexsort((arr_rows[:, 0], -arr_rows[:, 1]))
        arr_rows = arr_rows[order]
    else:
        arr_rows = np.empty((0, 7), dtype=np.float64)

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "lon_center",
            "lat_center",
            "mean_wsci",
            "std_wsci",
            "min_wsci",
            "max_wsci",
            "valid_pixel_count",
        ])
        for row in arr_rows:
            writer.writerow([
                f"{row[0]:.2f}",
                f"{row[1]:.2f}",
                f"{row[2]:.6f}",
                f"{row[3]:.6f}",
                f"{row[4]:.6f}",
                f"{row[5]:.6f}",
                int(round(row[6])),
            ])

    print(f"Saved 0.5-degree WSCI statistics CSV: {out_csv}")
    print(f"Number of non-empty 0.5-degree cells: {arr_rows.shape[0]}")


# =============================================================================
# Arctic-exclusion sensitivity diagnostics
# =============================================================================

EARTH_RADIUS_KM = 6371.0088


def raster_row_pixel_area_km2(transform, n_rows: int) -> np.ndarray:
    """Return spherical pixel area (km2) for each raster row."""
    lat_center = transform.f + (
        np.arange(n_rows, dtype=np.float64) + 0.5
    ) * transform.e
    half_dlat = 0.5 * abs(float(transform.e))
    lat1 = np.deg2rad(np.clip(lat_center - half_dlat, -90.0, 90.0))
    lat2 = np.deg2rad(np.clip(lat_center + half_dlat, -90.0, 90.0))
    dlon = np.deg2rad(abs(float(transform.a)))
    return (EARTH_RADIUS_KM ** 2) * dlon * np.abs(np.sin(lat2) - np.sin(lat1))


def compute_wsci_arctic_sensitivity(raster_path: str) -> Tuple[pd.DataFrame, dict]:
    """
    Compare WSCI diagnostics for the complete valid domain and latitude <=66N.

    Area-based quantities use spherical raster-cell area weighting. This is
    preferable to a raw pixel count when comparing high and low latitudes.
    """
    hist_edges = np.linspace(
        WSCI_HIST_RANGE[0],
        WSCI_HIST_RANGE[1],
        ARCTIC_DIAGNOSTIC_N_BINS + 1,
    )
    domain_keys = ["full", "lat_le_66"]
    hist_area = {
        key: np.zeros(ARCTIC_DIAGNOSTIC_N_BINS, dtype=np.float64)
        for key in domain_keys
    }
    acc = {
        key: {
            "valid_pixel_count": 0,
            "sensitive_pixel_count": 0,
            "valid_area_km2": 0.0,
            "sensitive_area_km2": 0.0,
            "area_weighted_wsci_sum": 0.0,
        }
        for key in domain_keys
    }

    with rasterio.open(raster_path) as src:
        nodata = src.nodata
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window, masked=False, out_dtype="float64")
            if nodata is not None and np.isfinite(nodata):
                arr[arr == nodata] = np.nan
            arr = normalize_raster_values(arr)
            transform = src.window_transform(window)

            lat_rows = transform.f + (
                np.arange(window.height, dtype=np.float64) + 0.5
            ) * transform.e
            row_area = raster_row_pixel_area_km2(transform, window.height)
            area2d = np.broadcast_to(row_area[:, None], arr.shape)

            valid_all = np.isfinite(arr)
            masks = {
                "full": valid_all,
                "lat_le_66": valid_all & (lat_rows[:, None] <= MAX_ANALYSIS_LAT),
            }

            for key, mask in masks.items():
                if not np.any(mask):
                    continue
                vals = arr[mask]
                weights = area2d[mask]
                sensitive = vals >= WSCI_HIST_THRESHOLD

                acc[key]["valid_pixel_count"] += int(vals.size)
                acc[key]["sensitive_pixel_count"] += int(np.sum(sensitive))
                acc[key]["valid_area_km2"] += float(np.sum(weights))
                acc[key]["sensitive_area_km2"] += float(np.sum(weights[sensitive]))
                acc[key]["area_weighted_wsci_sum"] += float(np.sum(vals * weights))

                in_hist = (
                    np.isfinite(vals)
                    & (vals >= hist_edges[0])
                    & (vals <= hist_edges[-1])
                )
                if np.any(in_hist):
                    hist_area[key] += np.histogram(
                        vals[in_hist],
                        bins=hist_edges,
                        weights=weights[in_hist],
                    )[0]

    rows = []
    for key in domain_keys:
        a = acc[key]
        valid_area = float(a["valid_area_km2"])
        valid_pixels = int(a["valid_pixel_count"])
        sensitive_area = float(a["sensitive_area_km2"])
        sensitive_pixels = int(a["sensitive_pixel_count"])
        rows.append(
            {
                "domain": key,
                "domain_label": (
                    ARCTIC_DOMAIN_LABEL_FULL
                    if key == "full"
                    else ARCTIC_DOMAIN_LABEL_CUTOFF
                ),
                "latitude_max_deg": np.nan if key == "full" else MAX_ANALYSIS_LAT,
                "valid_pixel_count": valid_pixels,
                "sensitive_pixel_count": sensitive_pixels,
                "valid_area_km2": valid_area,
                "sensitive_area_km2": sensitive_area,
                "sensitive_pixel_fraction_pct": (
                    100.0 * sensitive_pixels / valid_pixels
                    if valid_pixels > 0 else np.nan
                ),
                "sensitive_area_fraction_pct": (
                    100.0 * sensitive_area / valid_area
                    if valid_area > 0 else np.nan
                ),
                "area_weighted_mean_wsci": (
                    float(a["area_weighted_wsci_sum"]) / valid_area
                    if valid_area > 0 else np.nan
                ),
            }
        )

    summary = pd.DataFrame(rows)
    full_area = float(
        summary.loc[summary["domain"] == "full", "valid_area_km2"].iloc[0]
    )
    cutoff_area = float(
        summary.loc[summary["domain"] == "lat_le_66", "valid_area_km2"].iloc[0]
    )
    retained_pct = 100.0 * cutoff_area / full_area if full_area > 0 else np.nan
    summary["valid_area_retained_vs_full_pct"] = np.where(
        summary["domain"].eq("lat_le_66"),
        retained_pct,
        100.0,
    )

    diagnostics = {
        "hist_edges": hist_edges,
        "hist_area": hist_area,
    }
    return summary, diagnostics


def update_manuscript_summary_with_wsci(wsci_summary: pd.DataFrame) -> None:
    """Update the compact manuscript summary table without deleting wetland rows."""
    full = wsci_summary.loc[wsci_summary["domain"] == "full"].iloc[0]
    cut = wsci_summary.loc[wsci_summary["domain"] == "lat_le_66"].iloc[0]

    metric_specs = [
        ("Valid WSCI coastal area retained", 100.0, cut["valid_area_retained_vs_full_pct"], "% of full domain"),
        ("Wave-sensitive coastal area fraction", full["sensitive_area_fraction_pct"], cut["sensitive_area_fraction_pct"], "%"),
        ("Wave-sensitive pixel fraction", full["sensitive_pixel_fraction_pct"], cut["sensitive_pixel_fraction_pct"], "%"),
        ("Area-weighted mean WSCI", full["area_weighted_mean_wsci"], cut["area_weighted_mean_wsci"], "WSCI"),
    ]

    rows = []
    for metric, full_value, cutoff_value, unit in metric_specs:
        full_value = float(full_value)
        cutoff_value = float(cutoff_value)
        difference = cutoff_value - full_value
        relative = (
            100.0 * difference / abs(full_value)
            if np.isfinite(full_value) and abs(full_value) > 0
            else np.nan
        )
        rows.append(
            {
                "analysis": "WSCI",
                "group": "Global",
                "metric": metric,
                "full_domain_value": full_value,
                "lat_le_66N_value": cutoff_value,
                "difference": difference,
                "relative_change_pct": relative,
                "unit": unit,
            }
        )
    new_rows = pd.DataFrame(rows)

    if os.path.exists(MANUSCRIPT_SUMMARY_CSV):
        try:
            existing = pd.read_csv(MANUSCRIPT_SUMMARY_CSV)
            if "analysis" in existing.columns:
                existing = existing[existing["analysis"] != "WSCI"]
            combined = pd.concat([existing, new_rows], ignore_index=True)
        except Exception:
            combined = new_rows
    else:
        combined = new_rows

    combined.to_csv(MANUSCRIPT_SUMMARY_CSV, index=False)
    print(f"Updated manuscript summary table: {MANUSCRIPT_SUMMARY_CSV}")


def make_arctic_exclusion_wsci_figure(
    summary: pd.DataFrame,
    diagnostics: dict,
) -> None:
    """Create a compact Arctic-exclusion sensitivity figure with minimal text."""
    hist_edges = diagnostics["hist_edges"]
    centers = 0.5 * (hist_edges[:-1] + hist_edges[1:])

    fig, axes = plt.subplots(1, 2, figsize=ARCTIC_SENSITIVITY_FIGSIZE)

    domain_specs = [
        ("full", "All", "-"),
        ("lat_le_66", "≤66°N", "--"),
    ]

    # Panel a: area-weighted cumulative distribution.
    for key, label, linestyle in domain_specs:
        hist = np.asarray(diagnostics["hist_area"][key], dtype=float)
        total = np.nansum(hist)
        cdf = 100.0 * np.cumsum(hist) / total if total > 0 else np.full_like(hist, np.nan)
        axes[0].plot(
            centers,
            cdf,
            lw=ARCTIC_LINEWIDTH,
            ls=linestyle,
            label=label,
        )

    axes[0].axvline(
        WSCI_HIST_THRESHOLD,
        color="0.30",
        lw=0.95,
        ls=":",
    )
    axes[0].set_xlim(WSCI_HIST_RANGE)
    axes[0].set_ylim(0, 100)
    axes[0].set_xlabel("WSCI", fontsize=ARCTIC_AXIS_LABEL_FONT_SIZE)
    axes[0].set_ylabel("CDF (%)", fontsize=ARCTIC_AXIS_LABEL_FONT_SIZE)
    axes[0].legend(
        frameon=False,
        loc="upper left",
        fontsize=ARCTIC_LEGEND_FONT_SIZE,
        handlelength=2.0,
        borderaxespad=0.25,
        labelspacing=0.25,
    )
    axes[0].text(
        0.97,
        0.05,
        f"thr = {WSCI_HIST_THRESHOLD:.1f}",
        transform=axes[0].transAxes,
        ha="right",
        va="bottom",
        fontsize=ARCTIC_TITLE_FONT_SIZE,
        color="0.25",
    )
    axes[0].text(
        -0.095,
        1.02,
        "a",
        transform=axes[0].transAxes,
        fontsize=ARCTIC_PANEL_LABEL_FONT_SIZE,
        fontweight="bold",
        ha="left",
        va="bottom",
    )

    # Panel b: direct comparison of the wave-sensitive area fraction.
    order = ["full", "lat_le_66"]
    sub = summary.set_index("domain").loc[order]
    vals = sub["sensitive_area_fraction_pct"].to_numpy(dtype=float)
    x = np.arange(len(order))
    bars = axes[1].bar(
        x,
        vals,
        width=0.56,
        edgecolor="0.25",
        linewidth=0.65,
    )
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(["All", "≤66°N"], fontsize=ARCTIC_TICK_FONT_SIZE)
    axes[1].set_ylabel("Area (%)", fontsize=ARCTIC_AXIS_LABEL_FONT_SIZE)
    axes[1].set_title(f"WSCI ≥ {WSCI_HIST_THRESHOLD:.1f}", fontsize=ARCTIC_TITLE_FONT_SIZE, pad=4.0)
    axes[1].set_ylim(0, max(5.0, np.nanmax(vals) * 1.18))
    y_offset = 0.012 * axes[1].get_ylim()[1]
    for bar, val in zip(bars, vals):
        axes[1].text(
            bar.get_x() + bar.get_width() / 2.0,
            val + y_offset,
            f"{val:.1f}",
            ha="center",
            va="bottom",
            fontsize=ARCTIC_VALUE_FONT_SIZE,
        )
    axes[1].text(
        -0.095,
        1.02,
        "b",
        transform=axes[1].transAxes,
        fontsize=ARCTIC_PANEL_LABEL_FONT_SIZE,
        fontweight="bold",
        ha="left",
        va="bottom",
    )

    for ax in axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(
            direction="out",
            labelsize=ARCTIC_TICK_FONT_SIZE,
            length=3.0,
            width=0.7,
            pad=2.0,
        )

    fig.subplots_adjust(left=0.085, right=0.988, bottom=0.19, top=0.90, wspace=0.23)
    fig.savefig(
        ARCTIC_SENSITIVITY_OUTPUT,
        dpi=FIG_DPI,
        bbox_inches="tight",
        pad_inches=0.02,
        format="jpg",
        pil_kwargs={"quality": 96},
    )
    plt.close(fig)
    print(f"Saved Arctic-exclusion WSCI sensitivity figure: {ARCTIC_SENSITIVITY_OUTPUT}")

# =============================================================================
# Figure size and rendering
# =============================================================================

def estimate_main_figure_geometry() -> Tuple[Tuple[float, float], Tuple[float, float, float]]:
    """
    Compute a main-figure size that preserves the original local-panel aspect
    while expanding panel b from 2 x 3 to 3 x 3.

    Returns
    -------
    figsize : (width, height)
        Matplotlib figure size in inches.
    row_heights : (global_height, gap_height, local_block_height)
        Relative GridSpec heights, expressed in physical-inch-equivalent units.
    """
    usable_width = MAIN_FIGURE_WIDTH_INCH * (1.0 - 2.0 * MAIN_FIGURE_SIDE_MARGIN_FRAC)

    # Global Robinson map height.
    global_height_in = usable_width / ROBINSON_PANEL_WIDTH_TO_HEIGHT

    # Approximate local-panel width accounting for GridSpec wspace.
    local_panel_width = usable_width / (
        LOCAL_GRID_NCOLS + (LOCAL_GRID_NCOLS - 1) * LOCAL_GRID_WSPACE
    )
    local_panel_height = local_panel_width / LOCAL_PANEL_TARGET_RATIO

    # Preserve the same local-panel geometry after moving from 2 rows to 3 rows.
    local_block_height_in = (
        LOCAL_GRID_NROWS * local_panel_height
        + (LOCAL_GRID_NROWS - 1) * LOCAL_GRID_HSPACE * local_panel_height
    )

    gap_height_in = PANEL_A_TO_B_GAP_INCH
    content_height_in = global_height_in + gap_height_in + local_block_height_in

    usable_vertical_fraction = (
        MAIN_FIGURE_TOP_MARGIN_FRAC - MAIN_FIGURE_BOTTOM_MARGIN_FRAC
    )
    fig_height = content_height_in / usable_vertical_fraction

    return (
        (MAIN_FIGURE_WIDTH_INCH, fig_height),
        (global_height_in, gap_height_in, local_block_height_in),
    )


def create_figure() -> None:
    """Create the revised main Figure 5 and the standalone supplementary figure."""
    check_input_files()

    print("=" * 88)
    print("Creating revised WSCI Figure 5 and standalone supplementary river-flux figure")
    print("=" * 88)
    print(f"Input WSCI raster        : {INPUT_TIF}")
    print(f"Input sediment-flux CSV  : {CSV_FILE_PATH}")
    print(f"Output directory         : {OUTPUT_DIR}")
    print(f"Main Figure 5            : {MAIN_OUTPUT_PNG}")
    print(f"Supplementary figure     : {SUPPLEMENT_OUTPUT_PNG}")
    print(f"WSCI colormap             : {COLORMAP_CHOICE} -> {DATA_CMAP_NAME}")
    print(f"WSCI display range        : {DATA_VMIN} to {DATA_VMAX}")
    print(f"Primary latitude domain    : latitude <= {MAX_ANALYSIS_LAT:.1f} deg N")
    print()

    with rasterio.open(INPUT_TIF) as src:
        print("WSCI raster diagnostics:")
        print(f"  CRS          : {src.crs}")
        print(f"  Bounds       : {src.bounds}")
        print(f"  Size         : {src.width} x {src.height}")
        print(f"  Nodata       : {src.nodata}")
        print(f"  Transform    : {src.transform}")
        print(f"  Block shapes : {src.block_shapes}")
        print()

    print("Loading Natural Earth land and global/fallback rivers...")
    land_global = load_natural_earth_geometries(
        GLOBAL_NE_SCALE, "physical", NATURAL_EARTH_LAND_NAME
    )
    land_local = load_natural_earth_geometries(
        LOCAL_NE_SCALE, "physical", NATURAL_EARTH_LAND_NAME
    )
    rivers_global = load_natural_earth_geometries(
        GLOBAL_NE_SCALE, "physical", NATURAL_EARTH_RIVER_NAME
    )
    # Local Natural Earth rivers are retained only as a safety fallback.
    rivers_local_fallback = load_natural_earth_geometries(
        LOCAL_NE_SCALE, "physical", NATURAL_EARTH_RIVER_NAME
    )

    print(f"GRIT local-river GeoPackage: {GRIT_RIVER_GPKG_PATH}")
    grit_layer = detect_gpkg_layer(GRIT_RIVER_GPKG_PATH, GRIT_GPKG_LAYER)
    print(f"GRIT layer selected       : {grit_layer if grit_layer else '[default layer]'}")

    louisiana_rivers = []
    if os.path.exists(LOUISIANA_RIVER_SHP_PATH):
        print("Loading Louisiana-specific river shapefile...")
        louisiana_rivers = load_vector_geometries(LOUISIANA_RIVER_SHP_PATH)
    if len(louisiana_rivers) == 0:
        print("[WARNING] Louisiana custom rivers unavailable; panel 1 will use GRIT rivers.")
    print()

    # Preserve the original 0.5-degree statistics export.
    export_0p5deg_statistics(
        INPUT_TIF,
        OUTPUT_STATS_CSV,
        res_deg=STATS_RES_DEG,
    )
    print()

    # Quantify how excluding latitudes north of 66N changes the global WSCI
    # statistics. The primary Figure 5 already uses the <=66N domain.
    wsci_arctic_summary, wsci_arctic_diagnostics = compute_wsci_arctic_sensitivity(INPUT_TIF)
    wsci_arctic_summary.to_csv(ARCTIC_WSCI_SUMMARY_CSV, index=False)
    print(f"Saved Arctic-exclusion WSCI summary: {ARCTIC_WSCI_SUMMARY_CSV}")
    print(wsci_arctic_summary.to_string(index=False))
    make_arctic_exclusion_wsci_figure(
        wsci_arctic_summary,
        wsci_arctic_diagnostics,
    )
    update_manuscript_summary_with_wsci(wsci_arctic_summary)
    print()

    # Compute the river-flux statistics once; they are used only in the
    # standalone supplementary figure.
    df_flux = pd.read_csv(CSV_FILE_PATH)
    flux_stats = compute_flux_binned_statistics(df_flux)
    flux_stats.to_csv(OUTPUT_FLUX_STATS_CSV, index=False)
    print(f"Saved supplementary river-flux statistics CSV: {OUTPUT_FLUX_STATS_CSV}")
    print()

    # Build the moderate-resolution land mask used behind panel a.
    global_land_w = int(round(360.0 / GLOBAL_LAND_BACKGROUND_RES_DEG))
    global_land_h = int(round(180.0 / GLOBAL_LAND_BACKGROUND_RES_DEG))
    global_land_mask = rasterize_land_mask(
        land_global,
        extent=GLOBAL_EXTENT,
        shape=(global_land_h, global_land_w),
        all_touched=GLOBAL_LAND_MASK_ALL_TOUCHED,
    )

    # =========================================================================
    # MAIN FIGURE 5: panel a + panel b only
    # =========================================================================
    main_figsize, main_row_heights = estimate_main_figure_geometry()
    fig = plt.figure(figsize=main_figsize)
    fig.patch.set_facecolor(BACKGROUND_COLOR)

    # Explicit spacer row keeps the panel-a-to-panel-b gap easy to control.
    outer_gs = fig.add_gridspec(
        nrows=3,
        ncols=1,
        height_ratios=list(main_row_heights),
        hspace=0.0,
    )

    # Panel b is now a 3 x 3 local grid.
    local_gs = outer_gs[2, 0].subgridspec(
        nrows=LOCAL_GRID_NROWS,
        ncols=LOCAL_GRID_NCOLS,
        hspace=LOCAL_GRID_HSPACE,
        wspace=LOCAL_GRID_WSPACE,
    )

    fig.subplots_adjust(
        left=MAIN_FIGURE_SIDE_MARGIN_FRAC,
        right=1.0 - MAIN_FIGURE_SIDE_MARGIN_FRAC,
        top=MAIN_FIGURE_TOP_MARGIN_FRAC,
        bottom=MAIN_FIGURE_BOTTOM_MARGIN_FRAC,
    )

    # -------------------------------------------------------------------------
    # Panel a: global WSCI map
    # -------------------------------------------------------------------------
    ax_global = fig.add_subplot(outer_gs[0, 0])
    ax_global.set_facecolor(BACKGROUND_COLOR)

    hist_edges = np.linspace(
        WSCI_HIST_RANGE[0],
        WSCI_HIST_RANGE[1],
        WSCI_HIST_N_BINS + 1,
    )
    n_scatter, hist_counts = plot_global_native_scatter_robinson(
        ax_global,
        INPUT_TIF,
        hist_edges=hist_edges,
    )
    print(f"Global WSCI scatter points plotted: {n_scatter:,}")

    draw_global_land_fill(ax_global, global_land_mask)
    add_global_land_boundary(ax_global, global_land_mask)
    plot_global_rivers_robinson(ax_global, rivers_global)
    decorate_global_panel(ax_global, SITES)
    add_panel_label(ax_global, PANEL_A_LABEL, PANEL_A_LABEL_POS)
    add_wsci_histogram_cdf_inset(ax_global, hist_counts, hist_edges)

    ax_global.set_xticks([])
    ax_global.set_yticks([])
    for spine in ax_global.spines.values():
        spine.set_visible(False)

    sm = cm.ScalarMappable(norm=DATA_NORM, cmap=get_data_cmap())
    sm.set_array([])
    cax = ax_global.inset_axes(COLORBAR_INSET)
    cb = fig.colorbar(
        sm,
        cax=cax,
        orientation="horizontal",
        ticks=COLORBAR_TICKS,
    )
    cb.set_label(DATA_LABEL, fontsize=COLORBAR_LABEL_FONT_SIZE)
    cb.ax.tick_params(
        labelsize=COLORBAR_TICK_FONT_SIZE,
        length=2.0,
        width=0.6,
    )

    # -------------------------------------------------------------------------
    # Panel b: nine local zoom panels as one grouped 3 x 3 panel
    # -------------------------------------------------------------------------
    for i, site in enumerate(SITES):
        row = i // LOCAL_GRID_NCOLS
        col = i % LOCAL_GRID_NCOLS

        ax = fig.add_subplot(local_gs[row, col])
        ax.set_facecolor(BACKGROUND_COLOR)

        original_extent = site["extent"]
        display_extent = get_padded_local_extent(original_extent)

        print(f"Reading local panel {site['id']}: {site['name']}")
        print(f"  requested extent = {original_extent}")
        print(f"  display extent   = {display_extent}")

        local_data, actual_extent = read_native_window(INPUT_TIF, display_extent)
        print(f"  native read shape = {local_data.shape[1]} x {local_data.shape[0]}")
        print(f"  actual extent     = {actual_extent}")

        local_land_mask = rasterize_land_mask(
            land_local,
            extent=actual_extent,
            shape=local_data.shape,
            all_touched=LOCAL_LAND_MASK_ALL_TOUCHED,
        )
        local_data = mask_data_on_land(local_data, local_land_mask)

        ax.imshow(
            local_data,
            extent=actual_extent,
            origin="upper",
            cmap=get_data_cmap(),
            norm=DATA_NORM,
            interpolation="nearest",
            zorder=5,
        )
        draw_land_overlay_local(ax, local_land_mask, actual_extent)

        # ---------------------------------------------------------------------
        # Local river overlay
        # Louisiana retains the existing custom river shapefile because it
        # already performs well. All other zoom panels query GRIT by bbox.
        # If the Louisiana custom file is unavailable, Louisiana also uses GRIT.
        # Natural Earth is used only if a GRIT query unexpectedly returns empty.
        # ---------------------------------------------------------------------
        river_source_used = None

        if site.get("river_source") == "louisiana_custom" and len(louisiana_rivers) > 0:
            river_geoms = louisiana_rivers
            river_source_used = "Louisiana custom"
        else:
            river_geoms = load_grit_rivers_for_extent(actual_extent)
            river_source_used = "GRIT v1.0"

            if len(river_geoms) == 0:
                print(
                    f"  [WARNING] No GRIT river features found for site {site['id']} "
                    f"({site['name']}); falling back to Natural Earth."
                )
                river_geoms = rivers_local_fallback
                river_source_used = "Natural Earth fallback"

        n_river_parts = plot_local_rivers(ax, river_geoms, actual_extent)
        if SHOW_LOCAL_RIVER_DIAGNOSTICS:
            print(
                f"  river source     = {river_source_used}; "
                f"features={len(river_geoms):,}; line parts drawn={n_river_parts:,}"
            )

        draw_requested_extent_box(ax, original_extent)
        add_local_site_symbol(ax, site)
        add_local_site_number(ax, site)
        add_local_title(ax, site)

        ax.set_xlim(actual_extent[0], actual_extent[1])
        ax.set_ylim(actual_extent[2], actual_extent[3])
        apply_local_aspect(ax, actual_extent)

        if ALL_LOCAL_PANELS_USE_SCALEBAR_ONLY:
            show_x = False
            show_y = False
        elif SHOW_TICKS_ON_EVERY_LOCAL_PANEL:
            show_x = True
            show_y = True
        else:
            show_x = (row == LOCAL_GRID_NROWS - 1)
            show_y = (col == 0)

        set_local_ticks(
            ax,
            actual_extent,
            show_x=show_x,
            show_y=show_y,
        )

        if ALL_LOCAL_PANELS_USE_SCALEBAR_ONLY:
            add_local_scalebar(ax, actual_extent)

        if LABEL_LOCAL_BLOCK_ONLY:
            if i == 0:
                add_panel_label(ax, PANEL_B_LABEL, PANEL_B_LABEL_POS)
        else:
            add_panel_label(
                ax,
                LOCAL_PANEL_LABELS_WHEN_SEPARATE[i],
                PANEL_B_LABEL_POS,
            )

        del local_data, local_land_mask
        gc.collect()

    if SAVE_PNG:
        fig.savefig(
            MAIN_OUTPUT_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            pad_inches=0.025,
        )
        print(f"Saved main Figure 5: {MAIN_OUTPUT_PNG}")

    plt.close(fig)

    # =========================================================================
    # STANDALONE SUPPLEMENTARY FIGURE: former panel c, no panel letter
    # =========================================================================
    fig_sup, ax_flux = plt.subplots(
        figsize=(SUPPLEMENT_FIGURE_WIDTH_INCH, SUPPLEMENT_FIGURE_HEIGHT_INCH)
    )
    fig_sup.patch.set_facecolor(BACKGROUND_COLOR)
    ax_flux.set_facecolor(BACKGROUND_COLOR)

    plot_panel_c_river_sediment_flux(ax_flux, flux_stats)

    fig_sup.subplots_adjust(
        left=SUPPLEMENT_LEFT_MARGIN,
        right=SUPPLEMENT_RIGHT_MARGIN,
        bottom=SUPPLEMENT_BOTTOM_MARGIN,
        top=SUPPLEMENT_TOP_MARGIN,
    )

    if SAVE_PNG:
        fig_sup.savefig(
            SUPPLEMENT_OUTPUT_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            pad_inches=0.025,
        )
        print(f"Saved supplementary figure: {SUPPLEMENT_OUTPUT_PNG}")

    plt.close(fig_sup)

    write_runtime_notes()

    del global_land_mask
    gc.collect()


def write_runtime_notes() -> None:
    """Write a compact runtime note file for reproducibility."""
    with open(OUTPUT_NOTES, "w", encoding="utf-8") as f:
        f.write("Revised Figure 5 WSCI runtime notes\n")
        f.write("===================================\n")
        f.write(f"INPUT_TIF = {INPUT_TIF}\n")
        f.write(f"CSV_FILE_PATH = {CSV_FILE_PATH}\n")
        f.write(f"MAIN_OUTPUT_PNG = {MAIN_OUTPUT_PNG}\n")
        f.write(f"SUPPLEMENT_OUTPUT_PNG = {SUPPLEMENT_OUTPUT_PNG}\n")
        f.write(f"OUTPUT_STATS_CSV = {OUTPUT_STATS_CSV}\n")
        f.write(f"OUTPUT_FLUX_STATS_CSV = {OUTPUT_FLUX_STATS_CSV}\n")
        f.write(f"ARCTIC_SENSITIVITY_OUTPUT = {ARCTIC_SENSITIVITY_OUTPUT}\n")
        f.write(f"ARCTIC_WSCI_SUMMARY_CSV = {ARCTIC_WSCI_SUMMARY_CSV}\n")
        f.write(f"MAX_ANALYSIS_LAT = {MAX_ANALYSIS_LAT}\n")
        f.write(f"COLORMAP_CHOICE = {COLORMAP_CHOICE}\n")
        f.write(f"DATA_CMAP_NAME = {DATA_CMAP_NAME}\n")
        f.write(f"DATA_VMIN = {DATA_VMIN}\n")
        f.write(f"DATA_VMAX = {DATA_VMAX}\n")
        f.write(f"PANEL_A_LABEL = {PANEL_A_LABEL}\n")
        f.write(f"PANEL_B_LABEL = {PANEL_B_LABEL}\n")
        f.write(f"LOCAL_GRID_NROWS = {LOCAL_GRID_NROWS}\n")
        f.write(f"LOCAL_GRID_NCOLS = {LOCAL_GRID_NCOLS}\n")
        f.write(f"GLOBAL_SCATTER_PIXEL_STEP = {GLOBAL_SCATTER_PIXEL_STEP}\n")
        f.write(f"GLOBAL_SCATTER_SIZE = {GLOBAL_SCATTER_SIZE}\n")
        f.write(f"FONT_SCALE = {FONT_SCALE}\n")
        f.write(f"PANEL_A_TO_B_GAP_INCH = {PANEL_A_TO_B_GAP_INCH}\n")
        f.write(f"LOCAL_GRID_HSPACE = {LOCAL_GRID_HSPACE}\n")
        f.write(f"LOCAL_GRID_WSPACE = {LOCAL_GRID_WSPACE}\n")
        f.write(f"SHOW_WSCI_INSET = {SHOW_WSCI_INSET}\n")
        f.write(f"WSCI_INSET_AXES = {WSCI_INSET_AXES}\n")
        f.write(f"WSCI_HIST_RANGE = {WSCI_HIST_RANGE}\n")
        f.write(f"WSCI_HIST_N_BINS = {WSCI_HIST_N_BINS}\n")
        f.write(f"WSCI_HIST_THRESHOLD = {WSCI_HIST_THRESHOLD}\n")
        f.write(f"GLOBAL_NE_SCALE = {GLOBAL_NE_SCALE}\n")
        f.write(f"LOCAL_NE_SCALE = {LOCAL_NE_SCALE}\n")
        f.write(f"GLOBAL_RIVER_LINEWIDTH = {GLOBAL_RIVER_LINEWIDTH}\n")
        f.write(f"LOCAL_RIVER_LINEWIDTH = {LOCAL_RIVER_LINEWIDTH}\n")
        f.write(f"GLOBAL_LAND_BACKGROUND_RES_DEG = {GLOBAL_LAND_BACKGROUND_RES_DEG}\n")
        f.write(f"STATS_RES_DEG = {STATS_RES_DEG}\n")
        f.write(f"LOCAL_TITLE_FONT_SIZE = {LOCAL_TITLE_FONT_SIZE}\n")
        f.write(f"SHOW_LOCAL_SITE_NUMBER = {SHOW_LOCAL_SITE_NUMBER}\n")
        f.write(f"GLOBAL_BOX_EXPAND_FRACTION = {GLOBAL_BOX_EXPAND_FRACTION}\n")
        f.write(f"LOUISIANA_RIVER_SHP_PATH = {LOUISIANA_RIVER_SHP_PATH}\n")
        f.write(f"GRIT_RIVER_GPKG_PATH = {GRIT_RIVER_GPKG_PATH}\n")
        f.write(f"GRIT_GPKG_LAYER = {GRIT_GPKG_LAYER}\n")
        f.write(f"GRIT_BBOX_PAD_FRACTION = {GRIT_BBOX_PAD_FRACTION}\n")
        f.write(f"REQUIRE_GRIT_LOCAL_RIVERS = {REQUIRE_GRIT_LOCAL_RIVERS}\n")
        f.write("\nSites\n")
        f.write("-----\n")
        for site in SITES:
            f.write(
                f"{site['id']}. {site['name']} | "
                f"extent={site['extent']} | "
                f"river_source={site['river_source']}\n"
            )

    print(f"Saved notes: {OUTPUT_NOTES}")


def main() -> None:
    """Main driver."""
    create_figure()
    print("All requested revised Figure 5 outputs have been generated successfully.")


if __name__ == "__main__":
    main()


/scratch/7791788.1.mem384/ipykernel_720037/3758106911.py:344: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap(spec["name"])


Creating revised WSCI Figure 5 and standalone supplementary river-flux figure
Input WSCI raster        : /projectnb/nasa-marsh/Hangjie/cloud/saltmarsh_lossgain/processed/correlation_matrix_globalcolor_mask.tif
Input sediment-flux CSV  : /projectnb/nasa-marsh/Hangjie/cloud/sedimentflux_dethier/output_matrix.csv
Output directory         : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/
Main Figure 5            : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/Figure5_WSCI_global_zoom.png
Supplementary figure     : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/FigureS_WSCI_river_sediment_flux.png
WSCI colormap             : previous -> RdYlBu_r
WSCI display range        : 0.6 to 1.0
Primary latitude domain    : latitude <= 66.0 deg N

WSCI raster diagnostics:
  CRS          : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],